# Configuration


In [58]:
from pathlib import Path
from hashlib import sha256
import json
import re

import joblib
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from IPython.display import display


def locate_project_root():
    """Find the project root from the repository or notebooks directory."""
    here = Path.cwd().resolve()

    for candidate in (here, *here.parents):
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    return here.parent if here.name == "notebooks" else here


ROOT = locate_project_root()
DATA_DIR = ROOT / "data"

# Updated folder names.
TRAIN_DIR = DATA_DIR / "train"
VALIDATION_DIR = DATA_DIR / "validation"
SEALED_DIR = DATA_DIR / "sealed"

OUTPUT_DIR = ROOT / "outputs" / "stage4"
MODEL_DIR = ROOT / "models"

STAGE3_REFERENCE_MODEL_PATH = (
    MODEL_DIR / "stage3_fixed_reference.joblib"
)

STAGE3_REFERENCE_METADATA_PATH = (
    MODEL_DIR / "stage3_fixed_reference_metadata.json"
)


# Reproducibility
SEED = 42
np.random.seed(SEED)


# Challenge rules
STARTING_BALANCE = 5_000.0
MAX_DRAWDOWN_RATE = 0.04
DRAWDOWN_LIMIT_AMOUNT = STARTING_BALANCE * MAX_DRAWDOWN_RATE
REALIZED_DRAWDOWN_BOUNDARY = -DRAWDOWN_LIMIT_AMOUNT

PROFIT_TARGET_RATE = 0.08
PROFIT_TARGET_AMOUNT = STARTING_BALANCE * PROFIT_TARGET_RATE

# C22 confirmed a common campaign window from 9 AM to 9 AM SGT.
CHALLENGE_TIMEZONE = "Asia/Singapore"
CHALLENGE_START_HOUR = 9
CHALLENGE_DURATION_HOURS = 24


# Campaign splits
FIRST_TRAIN_CAMPAIGN = 33
INITIAL_TRAIN_END = 52
VALIDATION_START = 53
VALIDATION_END = 66
SEALED_START = 67
SEALED_END = 82

TRAIN_CAMPAIGNS = tuple(
    range(FIRST_TRAIN_CAMPAIGN, INITIAL_TRAIN_END + 1)
)

VALIDATION_CAMPAIGNS = tuple(
    range(VALIDATION_START, VALIDATION_END + 1)
)

SEALED_CAMPAIGNS = tuple(
    range(SEALED_START, SEALED_END + 1)
)


# Person identification
PERSON_ID_METHOD = "normalized_email_sha256"
MISSING_EMAIL_POLICY = "campaign_specific_fallback"


# Frozen 22-feature model specification
FEATURE_COLUMNS = [
    "post_loss_entry",
    "post_loss_reentry_gap_minutes",
    "post_loss_amount_ratio",
    "past_trade_open_count",
    "elapsed_active_hours",
    "past_trades_opened_per_active_hour",
    "trades_opened_past_30_minutes",
    "minutes_since_previous_trade_open",
    "past_median_trade_open_gap_minutes",
    "trade_open_gap_to_past_median_ratio",
    "minutes_since_previous_idea_start",
    "past_median_idea_start_gap_minutes",
    "idea_start_gap_to_past_median_ratio",
    "realized_distance_to_drawdown_limit",
    "historical_median_amount_before_trade",
    "current_to_historical_median_amount_ratio",
    "past_amount_cv",
    "past_completed_idea_count",
    "past_idea_win_rate",
    "past_mean_win_profit_per_lot",
    "past_mean_loss_abs_profit_per_lot",
    "past_payoff_ratio",
]

MODEL_PARAMETERS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 15,
    "random_state": SEED,
    "verbosity": -1,
}

QUANTILE = 0.90


# Inference and pass criteria
BOOTSTRAP_ITERATIONS = 2_000
CONFIDENCE_LEVEL = 0.95
MIN_POSITIVE_CAMPAIGN_SHARE = 0.60


# This is the only framework execution control.
ENABLED_FRAMEWORKS = [
    "walk_forward_baseline",
    "stage3_fixed_reference",
    "walk_forward_filtered"
]

AVAILABLE_FRAMEWORKS = {
    "walk_forward_baseline",
    "stage3_fixed_reference",
    "walk_forward_filtered",
}

unknown_frameworks = (
    set(ENABLED_FRAMEWORKS) - AVAILABLE_FRAMEWORKS
)

if unknown_frameworks:
    raise ValueError(
        "Unknown frameworks in ENABLED_FRAMEWORKS: "
        f"{sorted(unknown_frameworks)}"
    )

if not ENABLED_FRAMEWORKS:
    raise ValueError("At least one framework must be enabled.")


print("Project root:", ROOT)
print("Train directory:", TRAIN_DIR)
print("Validation directory:", VALIDATION_DIR)
print("Sealed directory:", SEALED_DIR)
print("Enabled frameworks:", ENABLED_FRAMEWORKS)
print("Frozen features:", len(FEATURE_COLUMNS))


Project root: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design
Train directory: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/train
Validation directory: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/validation
Sealed directory: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/sealed
Enabled frameworks: ['walk_forward_baseline', 'stage3_fixed_reference', 'walk_forward_filtered']
Frozen features: 22


## Input-directory validation


In [2]:
required_directories = {
    "train trades": TRAIN_DIR / "User Trades",
    "train user data": TRAIN_DIR / "User Data",
    "validation trades": VALIDATION_DIR / "User Trades",
    "validation user data": VALIDATION_DIR / "User Data",
    "sealed trades": SEALED_DIR / "User Trades",
    "sealed user data": SEALED_DIR / "User Data",
}

missing_directories = {
    label: path
    for label, path in required_directories.items()
    if not path.is_dir()
}

if missing_directories:
    missing_text = "\n".join(
        f"- {label}: {path}"
        for label, path in missing_directories.items()
    )

    raise FileNotFoundError(
        "Required input directories were not found:\n"
        f"{missing_text}\n\n"
        "Expected folder names are data/train, "
        "data/validation and data/sealed."
    )

print("All required input directories are available.")


All required input directories are available.


# Data loading helper functions


In [3]:
FILENAME_PATTERN = re.compile(
    r"Campaign[_\s]*(\d+)"
    r"[_\s]*Data[_\s]*"
    r"(\d{1,2}[_\s]*[A-Za-z]+[_\s]*\d{4})"
    r".*?"
    r"(Traders[\s_]*only|XAUUSD[\s_]*only)",
    re.IGNORECASE,
)


TRADE_ID_COLUMNS = [
    "account_id",
    "close_trade_id",
    "position_id",
    "close_order_id",
    "open_order_id",
    "user_group_id",
]


TRADE_DATETIME_COLUMNS = [
    "open_date_time",
    "close_date_time",
]


TRADE_NUMERIC_COLUMNS = [
    "lot_size",
    "duration_sec",
    "profit",
    "reverse_profit",
    "net_profit",
    "commission",
    "swap",
    "amount",
    "open_price",
    "close_price",
    "sl_price",
    "tp_price",
    "open_trade_cross_price",
    "close_trade_cross_price",
]


TRADE_COLUMN_ALIASES = {
    "account_id": "account_id",
    "accountid": "account_id",

    "close_trade_id": "close_trade_id",
    "closetradeid": "close_trade_id",

    "position_id": "position_id",
    "positionid": "position_id",

    "close_order_id": "close_order_id",
    "closeorderid": "close_order_id",

    "open_order_id": "open_order_id",
    "openorderid": "open_order_id",

    "user_group_id": "user_group_id",
    "usergroupid": "user_group_id",

    "open_date_time": "open_date_time",
    "opendatetime": "open_date_time",

    "close_date_time": "close_date_time",
    "closedatetime": "close_date_time",

    "lot_size": "lot_size",
    "lotsize": "lot_size",

    "duration_sec": "duration_sec",
    "durationsec": "duration_sec",

    "profit": "profit",

    "reverse_profit": "reverse_profit",
    "reverseprofit": "reverse_profit",

    "net_profit": "net_profit",
    "netprofit": "net_profit",

    "commission": "commission",
    "swap": "swap",
    "amount": "amount",

    "open_price": "open_price",
    "openprice": "open_price",

    "close_price": "close_price",
    "closeprice": "close_price",

    "sl_price": "sl_price",
    "slprice": "sl_price",

    "tp_price": "tp_price",
    "tpprice": "tp_price",

    "open_trade_cross_price":
        "open_trade_cross_price",
    "opentradecrossprice":
        "open_trade_cross_price",

    "close_trade_cross_price":
        "close_trade_cross_price",
    "closetradecrossprice":
        "close_trade_cross_price",

    "side": "side",
    "currency": "currency",

    "campaign_id": "source_campaign_id",
    "campaignid": "source_campaign_id",
}


In [4]:
def normalize_column_name(
    column,
):
    """Convert a raw column name to lowercase snake case."""

    column_name = str(
        column
    ).strip()

    column_name = re.sub(
        r"([A-Z]+)([A-Z][a-z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name,
    )

    column_name = re.sub(
        r"_+",
        "_",
        column_name,
    )

    return (
        column_name
        .strip("_")
        .lower()
    )


In [5]:
def parse_campaign_date(
    date_string,
):
    """Parse the campaign date encoded in a filename."""

    normalized_date_string = re.sub(
        r"[_\s]+",
        " ",
        date_string.strip(),
    )

    for date_format in [
        "%d %b %Y",
        "%d %B %Y",
    ]:
        parsed_date = pd.to_datetime(
            normalized_date_string,
            format=date_format,
            errors="coerce",
        )

        if pd.notna(
            parsed_date
        ):
            return parsed_date

    return pd.NaT


In [6]:
def parse_filename(
    path,
):
    """Extract campaign metadata from a raw trade filename."""

    path = Path(
        path
    )

    filename = (
        path.stem
    )

    match = (
        FILENAME_PATTERN
        .search(
            filename
        )
    )

    if match:
        return {
            "campaign_id":
                int(
                    match.group(1)
                ),
            "campaign_date":
                parse_campaign_date(
                    match.group(2)
                ),
        }

    campaign_match = re.search(
        r"Campaign[_\s]*(\d+)",
        filename,
        re.IGNORECASE,
    )

    campaign_id = (
        int(
            campaign_match.group(1)
        )
        if campaign_match
        else None
    )

    return {
        "campaign_id":
            campaign_id,
        "campaign_date":
            pd.NaT,
    }


In [7]:
def load_trade_file(
    path,
):
    """Load and standardize one raw campaign trade file."""

    path = Path(
        path
    )

    metadata = (
        parse_filename(
            path
        )
    )

    if (
        metadata[
            "campaign_id"
        ]
        is None
    ):
        raise ValueError(
            "Could not determine campaign ID "
            f"from {path.name}."
        )

    if (
        path.suffix.lower()
        == ".csv"
    ):
        df = pd.read_csv(
            path,
            low_memory=False,
        )

    elif (
        path.suffix.lower()
        == ".xlsx"
    ):
        df = pd.read_excel(
            path
        )

    else:
        raise ValueError(
            f"Unsupported file type: "
            f"{path.suffix}"
        )

    normalized_columns = {
        column:
            normalize_column_name(
                column
            )
        for column in df.columns
    }

    df = df.rename(
        columns=normalized_columns
    )

    df = df.rename(
        columns=TRADE_COLUMN_ALIASES
    )

    df.insert(
        0,
        "source_row_number",
        np.arange(
            2,
            len(df) + 2,
        ),
    )

    required_columns = {
        "account_id",
        "open_date_time",
        "amount",
        "side",
    }

    missing_required = (
        required_columns
        - set(
            df.columns
        )
    )

    if missing_required:
        raise ValueError(
            f"{path.name} is missing "
            "required columns: "
            f"{sorted(missing_required)}"
        )

    if "close_date_time" not in df.columns:
        df["close_date_time"] = pd.NaT
    if "net_profit" not in df.columns:
        df["net_profit"] = np.nan
    if "reverse_profit" not in df.columns:
        df["reverse_profit"] = np.nan

    header_echo_mask = (
        df[
            "account_id"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "account_id",
                "accountid",
                "account",
            }
        )
        .fillna(False)
        |
        df[
            "open_date_time"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "open_date_time",
                "opendatetime",
            }
        )
        .fillna(False)
    )

    df = (
        df.loc[
            ~header_echo_mask
        ]
        .copy()
    )

    for column in (
        TRADE_ID_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace(
                    {
                        "":
                            pd.NA,
                        "nan":
                            pd.NA,
                        "None":
                            pd.NA,
                    }
                )
            )

    for column in (
        TRADE_NUMERIC_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_numeric(
                df[column],
                errors="coerce",
            )

    for column in (
        TRADE_DATETIME_COLUMNS
    ):
        if (
            column
            in df.columns
        ):
            df[
                column
            ] = pd.to_datetime(
                df[column],
                errors="coerce",
                utc=True,
            )

    df[
        "side"
    ] = (
        df[
            "side"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    if (
        "currency"
        in df.columns
    ):
        df[
            "currency"
        ] = (
            df[
                "currency"
            ]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    df[
        "campaign_id"
    ] = metadata[
        "campaign_id"
    ]

    df[
        "campaign_date"
    ] = metadata[
        "campaign_date"
    ]

    df[
        "source_file"
    ] = path.name

    df[
        "source_path"
    ] = str(
        path
    )

    return (
        df.reset_index(
            drop=True
        )
    )


In [8]:
def load_trade_directory(
    directory,
    is_unseen,
):
    """Load every supported trade file in a directory."""

    directory = Path(
        directory
    )

    dataset_label = (
        "UNSEEN"
        if is_unseen
        else "HISTORICAL"
    )

    print(
        f"\nScanning {dataset_label.lower()} "
        f"trade directory:"
    )
    print(
        f"  {directory}"
    )

    if not directory.exists():
        raise FileNotFoundError(
            f"Trade directory not found: "
            f"{directory.resolve()}"
        )

    paths = sorted(
        [
            path
            for path in directory.rglob("*")
            if (
                path.is_file()
                and path.suffix.lower()
                in {
                    ".csv",
                    ".xlsx",
                }
            )
        ]
    )

    if not paths:
        raise ValueError(
            "No CSV or XLSX trade files "
            f"found in {directory}."
        )

    print(
        f"Found {len(paths)} "
        f"{dataset_label.lower()} files."
    )

    frames = []

    for file_number, path in enumerate(
        paths,
        start=1,
    ):
        print(
            f"  [{file_number}/{len(paths)}] "
            f"Processing: {path.name}"
        )

        frame = (
            load_trade_file(
                path
            )
        )

        frame[
            "_is_unseen"
        ] = bool(
            is_unseen
        )

        frames.append(
            frame
        )

    combined = pd.concat(
        frames,
        ignore_index=True,
    )

    print(
        f"Loaded {len(combined):,} "
        f"{dataset_label.lower()} trades."
    )

    return combined


## User Data loading and identity-key normalization


In [9]:
def campaign_from_path(path):
    """Extract a campaign ID from a filename or parent directory."""
    path = Path(path)

    for part in (path.stem, *(parent.name for parent in path.parents)):
        match = re.search(
            r"(?:Campaign[_\s-]*|\bC)(\d{2,3})(?!\d)",
            part,
            re.IGNORECASE,
        )

        if match:
            return int(match.group(1))

    raise ValueError(
        f"Cannot determine campaign ID from {path}."
    )


def normalise_account(values):
    """Normalise account IDs for campaign-specific joins."""
    result = (
        values.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    return result.mask(
        result.str.lower().isin(
            ["", "nan", "none", "null", "<na>"]
        )
    )


def normalise_email(values):
    """Normalise emails while preserving missing values."""
    result = (
        values.astype("string")
        .str.strip()
        .str.lower()
    )

    return result.mask(
        result.isin(
            ["", "nan", "none", "null", "<na>"]
        )
    )


def load_user_directory(directory):
    """Load and normalise User Data without constructing identities."""
    directory = Path(directory)

    if not directory.is_dir():
        raise FileNotFoundError(
            f"Missing User Data directory: {directory}"
        )

    paths = sorted(
        path
        for path in directory.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in {".csv", ".xlsx"}
            and not path.name.startswith("~$")
        )
    )

    if not paths:
        raise ValueError(
            f"No User Data files found in {directory}"
        )

    frames = []

    print(f"Loading {len(paths)} User Data files from {directory}...")

    for index, path in enumerate(paths, start=1):

        # Load file.
        if path.suffix.lower() == ".csv":
            frame = pd.read_csv(
                path,
                dtype="string",
                low_memory=False,
            )
        else:
            frame = pd.read_excel(
                path,
                dtype="string",
            )

        # Normalise column names.
        frame.columns = [
            normalize_column_name(column)
            for column in frame.columns
        ]

        # Handle known User Data aliases.
        frame = frame.rename(
            columns={
                "account": "account_id",
                "accountid": "account_id",
                "emailaddress": "email",
                "ipaddress": "ip_address",
            }
        )

        # Reject ambiguous columns.
        if frame.columns.duplicated().any():
            duplicated = (
                frame.columns[
                    frame.columns.duplicated()
                ].tolist()
            )

            raise ValueError(
                f"{path.name}: duplicated columns "
                f"after normalisation: {duplicated}"
            )

        required_columns = {
            "account_id",
            "email",
        }

        missing_columns = (
            required_columns - set(frame.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{path.name}: missing required columns "
                f"{sorted(missing_columns)}. "
                f"Available columns: {frame.columns.tolist()}"
            )

        # Retain only fields needed for identity mapping.
        frame = frame[
            ["account_id", "email"]
        ].copy()

        frame["account_id"] = normalise_account(
            frame["account_id"]
        )

        frame["email"] = normalise_email(
            frame["email"]
        )

        frame["campaign_id"] = campaign_from_path(
            path
        )

        frames.append(frame)

        print(
            f"  [{index}/{len(paths)}] "
            f"Campaign {frame['campaign_id'].iloc[0]}: "
            f"{len(frame):,} user records"
        )

    users = pd.concat(
        frames,
        ignore_index=True,
    )

    print(
        f"Total User Data rows loaded: {len(users):,}"
    )

    return users


# Load train and validation data


In [10]:
def validate_campaign_range(
    data,
    expected_campaigns,
    label,
):
    """Verify that a dataset contains exactly the expected campaigns."""
    expected = set(expected_campaigns)

    actual = set(
        data["campaign_id"]
        .dropna()
        .astype(int)
        .unique()
    )

    if actual != expected:
        raise ValueError(
            f"{label}: "
            f"missing campaigns {sorted(expected - actual)}; "
            f"unexpected campaigns {sorted(actual - expected)}."
        )

    print(
        f"{label}: {len(actual)} campaigns verified "
        f"({min(actual)}–{max(actual)})."
    )


print("Loading historical User Data...")

train_users = load_user_directory(
    TRAIN_DIR / "User Data"
)

validation_users = load_user_directory(
    VALIDATION_DIR / "User Data"
)

validate_campaign_range(
    train_users,
    TRAIN_CAMPAIGNS,
    "Train User Data",
)

validate_campaign_range(
    validation_users,
    VALIDATION_CAMPAIGNS,
    "Validation User Data",
)

historical_user_records = pd.concat(
    [train_users, validation_users],
    ignore_index=True,
)


print("\nLoading historical trades...")

train_trades = load_trade_directory(
    TRAIN_DIR / "User Trades",
    is_unseen=False,
)

validation_trades = load_trade_directory(
    VALIDATION_DIR / "User Trades",
    is_unseen=False,
)

validate_campaign_range(
    train_trades,
    TRAIN_CAMPAIGNS,
    "Train trades",
)

validate_campaign_range(
    validation_trades,
    VALIDATION_CAMPAIGNS,
    "Validation trades",
)

historical_trades_raw = pd.concat(
    [train_trades, validation_trades],
    ignore_index=True,
)

historical_trades_raw["account_id"] = normalise_account(
    historical_trades_raw["account_id"]
)

if historical_trades_raw["account_id"].isna().any():
    raise ValueError(
        "Historical trades contain missing account IDs."
    )

print("Historical User Data rows:", f"{len(historical_user_records):,}")
print("Historical trade rows:", f"{len(historical_trades_raw):,}")
print(
    "Historical campaigns:",
    historical_trades_raw["campaign_id"].nunique(),
)


Loading historical User Data...
Loading 20 User Data files from /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/train/User Data...
  [1/20] Campaign 33: 500 user records
  [2/20] Campaign 34: 281 user records
  [3/20] Campaign 35: 500 user records
  [4/20] Campaign 36: 401 user records
  [5/20] Campaign 37: 455 user records
  [6/20] Campaign 38: 500 user records
  [7/20] Campaign 39: 340 user records
  [8/20] Campaign 40: 381 user records
  [9/20] Campaign 41: 223 user records
  [10/20] Campaign 42: 500 user records
  [11/20] Campaign 43: 500 user records
  [12/20] Campaign 44: 500 user records
  [13/20] Campaign 45: 500 user records
  [14/20] Campaign 46: 500 user records
  [15/20] Campaign 47: 500 user records
  [16/20] Campaign 48: 500 user records
  [17/20] Campaign 49: 500 user records
  [18/20] Campaign 50: 500 user records
  [19/20] Campaign 51: 500 user records
  [20/20] Campaign 52: 500 user records
Total User Data rows loaded: 9,081
Load

# Build and audit person_id


In [39]:
def make_person_id(
    email,
    campaign_id,
    account_id,
):
    """
    Generate a consistent email-based person ID.

    If email is unavailable, use a campaign-specific fallback
    so reused account IDs are never linked across campaigns.
    """
    if pd.notna(email):
        value = f"email:{email}"

        return (
            "email_"
            + sha256(
                value.encode("utf-8")
            ).hexdigest()
        )

    value = (
        f"missing_identity:"
        f"{int(campaign_id)}:"
        f"{account_id}"
    )

    return (
        "fallback_"
        + sha256(
            value.encode("utf-8")
        ).hexdigest()
    )


def audit_user_identities(
    users,
    trades,
    expected_campaigns,
    dataset_label,
):
    """
    Audit User Data and map every trading account to a person.

    User Data is joined to trades using campaign_id and account_id
    within each campaign. Normalized email is used as the consistent
    person identifier across campaigns.
    """
    users = users.copy()
    trades = trades.copy()

    expected_campaigns = set(
        int(campaign_id)
        for campaign_id in expected_campaigns
    )

    join_columns = [
        "campaign_id",
        "account_id",
    ]

    print("=" * 60)
    print(
        f"{dataset_label.upper()} USER IDENTITY AUDIT"
    )
    print("=" * 60)

    # ==================================================
    # 1. Validate and normalize input keys
    # ==================================================

    for frame, frame_label in [
        (users, "User Data"),
        (trades, "User Trades"),
    ]:
        required_columns = set(
            join_columns
        )

        if frame_label == "User Data":
            required_columns.add(
                "email"
            )

        missing_columns = (
            required_columns
            - set(frame.columns)
        )

        if missing_columns:
            raise ValueError(
                f"{dataset_label} {frame_label}: "
                f"missing columns "
                f"{sorted(missing_columns)}"
            )

        if frame[
            "campaign_id"
        ].isna().any():
            raise ValueError(
                f"{dataset_label} {frame_label}: "
                "missing campaign IDs."
            )

        frame["campaign_id"] = (
            frame["campaign_id"]
            .astype(int)
        )

        frame["account_id"] = (
            normalise_account(
                frame["account_id"]
            )
        )

        if frame[
            "account_id"
        ].isna().any():
            raise ValueError(
                f"{dataset_label} {frame_label}: "
                "missing account IDs."
            )

    users["email"] = normalise_email(
        users["email"]
    )

    # ==================================================
    # 2. Validate campaign coverage
    # ==================================================

    user_campaigns = set(
        users["campaign_id"]
        .dropna()
        .astype(int)
        .unique()
    )

    trade_campaigns = set(
        trades["campaign_id"]
        .dropna()
        .astype(int)
        .unique()
    )

    if user_campaigns != expected_campaigns:
        raise ValueError(
            f"{dataset_label} User Data campaign mismatch. "
            f"Missing campaigns: "
            f"{sorted(expected_campaigns - user_campaigns)}. "
            f"Unexpected campaigns: "
            f"{sorted(user_campaigns - expected_campaigns)}."
        )

    if trade_campaigns != expected_campaigns:
        raise ValueError(
            f"{dataset_label} trade campaign mismatch. "
            f"Missing campaigns: "
            f"{sorted(expected_campaigns - trade_campaigns)}. "
            f"Unexpected campaigns: "
            f"{sorted(trade_campaigns - expected_campaigns)}."
        )

    print(
        f"Campaigns verified: "
        f"{min(expected_campaigns)}–"
        f"{max(expected_campaigns)}."
    )

    # ==================================================
    # 3. Audit duplicate and conflicting User Data
    # ==================================================

    original_user_rows = len(
        users
    )

    missing_email_rows = int(
        users["email"].isna().sum()
    )

    duplicate_rows = int(
        users.duplicated(
            subset=join_columns
        ).sum()
    )

    email_counts = (
        users.groupby(
            join_columns
        )["email"]
        .nunique(
            dropna=True
        )
    )

    conflicting_email_mappings = int(
        email_counts.gt(1).sum()
    )

    email_presence_counts = (
        users.assign(
            email_missing=(
                users["email"].isna()
            )
        )
        .groupby(
            join_columns
        )["email_missing"]
        .nunique()
    )

    inconsistent_email_mappings = int(
        email_presence_counts
        .gt(1)
        .sum()
    )

    # ==================================================
    # 4. Cross-campaign identity audit
    # ==================================================

    valid_users = (
        users.dropna(
            subset=["email"]
        )
        .copy()
    )

    # Same account ID associated with different emails.
    account_email_counts = (
        valid_users.groupby(
            "account_id"
        )["email"]
        .nunique()
    )

    reused_account_ids = int(
        account_email_counts
        .gt(1)
        .sum()
    )

    # Same email associated with multiple account IDs.
    email_account_counts = (
        valid_users.groupby(
            "email"
        )["account_id"]
        .nunique()
    )

    emails_multiple_accounts = int(
        email_account_counts
        .gt(1)
        .sum()
    )

    # Same email associated with multiple accounts
    # within one campaign.
    campaign_email_counts = (
        valid_users.groupby(
            [
                "campaign_id",
                "email",
            ]
        )["account_id"]
        .nunique()
    )

    same_campaign_shared_emails = int(
        campaign_email_counts
        .gt(1)
        .sum()
    )

    print("\nUser Data checks:")
    print(
        "Raw user records:",
        original_user_rows,
    )
    print(
        "Missing email rows:",
        missing_email_rows,
    )
    print(
        "Duplicate mapping rows:",
        duplicate_rows,
    )

    # ==================================================
    # 5. Reject ambiguous mappings
    # ==================================================

    if conflicting_email_mappings > 0:
        raise ValueError(
            f"{dataset_label}: "
            f"{conflicting_email_mappings} "
            "campaign-account pairs map to "
            "conflicting emails."
        )

    if inconsistent_email_mappings > 0:
        raise ValueError(
            f"{dataset_label}: "
            f"{inconsistent_email_mappings} "
            "campaign-account pairs contain both "
            "missing and present emails."
        )

    # Remaining duplicate rows agree on email.
    users = (
        users.drop_duplicates(
            subset=join_columns
        )
        .copy()
    )

    # ==================================================
    # 6. Identify accounts that actually traded
    # ==================================================

    trading_accounts = (
        trades[
            join_columns
        ]
        .drop_duplicates()
        .copy()
    )

    if trading_accounts.duplicated(
        join_columns
    ).any():
        raise AssertionError(
            "Trading account keys must be unique."
        )

    if users.duplicated(
        join_columns
    ).any():
        raise AssertionError(
            "User Data mapping keys must be unique."
        )

    print(
        "\nUnique trading campaign-account pairs:",
        len(trading_accounts),
    )

    # ==================================================
    # 7. Join trading accounts to User Data
    # ==================================================

    # Left join preserves trading accounts that are
    # completely absent from User Data.
    mapping = trading_accounts.merge(
        users[
            join_columns
            + ["email"]
        ],
        on=join_columns,
        how="left",
        indicator=True,
        validate="one_to_one",
    )

    missing_user_mask = (
        mapping["_merge"]
        .eq("left_only")
    )

    missing_email_mask = (
        mapping["_merge"].eq("both")
        & mapping["email"].isna()
    )

    email_based_mask = (
        mapping["_merge"].eq("both")
        & mapping["email"].notna()
    )

    mapping["identity_status"] = (
        "missing_user_record_fallback"
    )

    mapping.loc[
        missing_email_mask,
        "identity_status",
    ] = "missing_email_fallback"

    mapping.loc[
        email_based_mask,
        "identity_status",
    ] = "email_based"

    mapping["person_id"] = [
        make_person_id(
            email=row.email,
            campaign_id=row.campaign_id,
            account_id=row.account_id,
        )
        for row in mapping.itertuples(
            index=False
        )
    ]

    # ==================================================
    # 8. Missing-email accounts without trades
    # ==================================================

    user_usage = users[
        join_columns
        + ["email"]
    ].merge(
        trading_accounts.assign(
            has_trades=True
        ),
        on=join_columns,
        how="left",
        validate="one_to_one",
    )

    user_usage["has_trades"] = (
        user_usage["has_trades"]
        .fillna(False)
        .astype(bool)
    )

    missing_email_users = (
        user_usage.loc[
            user_usage[
                "email"
            ].isna()
        ]
        .copy()
    )

    missing_email_summary = (
        missing_email_users.groupby(
            "campaign_id"
        )
        .agg(
            missing_email_accounts=(
                "account_id",
                "size",
            ),
            accounts_with_trades=(
                "has_trades",
                "sum",
            ),
        )
        .reset_index()
        .sort_values(
            "campaign_id"
        )
    )

    missing_email_summary[
        "accounts_without_trades"
    ] = (
        missing_email_summary[
            "missing_email_accounts"
        ]
        - missing_email_summary[
            "accounts_with_trades"
        ]
    )

    # ==================================================
    # 9. Count trades affected by missing email
    # ==================================================

    missing_email_keys = (
        mapping.loc[
            missing_email_mask,
            join_columns,
        ]
        .copy()
    )

    missing_email_trades = trades.merge(
        missing_email_keys,
        on=join_columns,
        how="inner",
        validate="many_to_one",
    )

    # ==================================================
    # 10. Count trades affected by missing User Data
    # ==================================================

    missing_user_keys = (
        mapping.loc[
            missing_user_mask,
            join_columns,
        ]
        .copy()
    )

    missing_user_trades = trades.merge(
        missing_user_keys,
        on=join_columns,
        how="inner",
        validate="many_to_one",
    )

    missing_user_summary = (
        missing_user_trades.groupby(
            "campaign_id"
        )
        .agg(
            affected_trade_rows=(
                "account_id",
                "size",
            ),
            affected_accounts=(
                "account_id",
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            "campaign_id"
        )
    )

    # ==================================================
    # 11. Construct audit summary
    # ==================================================

    identity_audit = pd.DataFrame(
        [
            {
                "dataset": dataset_label,

                "user_data_rows":
                    original_user_rows,

                "unique_trading_accounts":
                    len(trading_accounts),

                "duplicate_user_data_rows":
                    duplicate_rows,

                "account_ids_reused_by_different_emails":
                    reused_account_ids,

                "emails_using_multiple_account_ids":
                    emails_multiple_accounts,

                "same_email_multiple_accounts_in_campaign":
                    same_campaign_shared_emails,

                "missing_email_user_data_rows":
                    missing_email_rows,

                "missing_email_accounts_with_trades":
                    int(
                        missing_email_mask.sum()
                    ),

                "missing_email_accounts_without_trades":
                    int(
                        (
                            ~missing_email_users[
                                "has_trades"
                            ]
                        ).sum()
                    ),

                "missing_email_affected_trade_rows":
                    len(
                        missing_email_trades
                    ),

                "missing_user_record_accounts":
                    int(
                        missing_user_mask.sum()
                    ),

                "missing_user_record_affected_trade_rows":
                    len(
                        missing_user_trades
                    ),

                "email_based_account_mappings":
                    int(
                        email_based_mask.sum()
                    ),

                "unique_person_ids":
                    mapping[
                        "person_id"
                    ].nunique(),
            }
        ]
    )

    # ==================================================
    # 12. Validate final identity mapping
    # ==================================================

    if len(mapping) != len(
        trading_accounts
    ):
        raise AssertionError(
            "Identity mapping does not cover "
            "all trading accounts."
        )

    if mapping[
        "person_id"
    ].isna().any():
        raise AssertionError(
            "Some trading accounts have "
            "missing person IDs."
        )

    if mapping.duplicated(
        join_columns
    ).any():
        raise AssertionError(
            "Duplicate campaign-account "
            "mappings found."
        )

    identity_table = mapping[
        join_columns
        + [
            "person_id",
            "identity_status",
        ]
    ].copy()

    # ==================================================
    # 13. Display results
    # ==================================================

    print(
        "\n"
        + "=" * 60
    )
    print(
        f"FINAL {dataset_label.upper()} "
        "IDENTITY AUDIT"
    )
    print("=" * 60)

    print("\nCombined audit:")
    display(
        identity_audit
    )

    print(
        "\nMissing-email accounts by campaign:"
    )
    display(
        missing_email_summary
    )

    print(
        "\nMissing User Data accounts by campaign:"
    )
    display(
        missing_user_summary
    )

    print(
        "\nIdentity status counts:"
    )

    identity_status_summary = (
        identity_table[
            "identity_status"
        ]
        .value_counts()
        .rename_axis(
            "identity_status"
        )
        .reset_index(
            name="account_mappings"
        )
    )

    display(
        identity_status_summary
    )

    print(
        "\nIdentity mapping completed:",
        f"{len(identity_table):,} "
        "trading accounts.",
    )

    return (
        identity_table,
        identity_audit,
        missing_email_summary,
        missing_user_summary,
    )


# ======================================================
# Run the historical identity audit
# ======================================================

(
    historical_users,
    identity_audit,
    missing_email_summary,
    missing_user_summary,
) = audit_user_identities(
    users=historical_user_records,
    trades=historical_trades_raw,
    expected_campaigns=(
        TRAIN_CAMPAIGNS
        + VALIDATION_CAMPAIGNS
    ),
    dataset_label="Historical",
)

print(
    "\nHistorical identity audit completed."
)

HISTORICAL USER IDENTITY AUDIT
Campaigns verified: 33–66.

User Data checks:
Raw user records: 15878
Missing email rows: 391
Duplicate mapping rows: 3

Unique trading campaign-account pairs: 8165

FINAL HISTORICAL IDENTITY AUDIT

Combined audit:


,dataset,user_data_rows,unique_trading_accounts,duplicate_user_data_rows,account_ids_reused_by_different_emails,emails_using_multiple_account_ids,same_email_multiple_accounts_in_campaign,missing_email_user_data_rows,missing_email_accounts_with_trades,missing_email_accounts_without_trades,missing_email_affected_trade_rows,missing_user_record_accounts,missing_user_record_affected_trade_rows,email_based_account_mappings,unique_person_ids
0,Historical,15878,8165,3,500,2523,57,391,23,368,116,6,28,8136,3579



Missing-email accounts by campaign:


,campaign_id,missing_email_accounts,accounts_with_trades,accounts_without_trades
0,44,65,11,54
1,50,56,3,53
2,51,136,0,136
3,56,77,9,68
4,59,57,0,57



Missing User Data accounts by campaign:


,campaign_id,affected_trade_rows,affected_accounts
0,33,10,3
1,34,1,1
2,37,17,2



Identity status counts:


,identity_status,account_mappings
0,email_based,8136
1,missing_email_fallback,23
2,missing_user_record_fallback,6



Identity mapping completed: 8,165 trading accounts.

Historical identity audit completed.


In [15]:
def attach_person_ids(trades, identity_table):
    """Attach identities within campaigns without dropping trade rows."""
    join_columns = [
        "campaign_id",
        "account_id",
    ]

    trades = trades.copy()

    # Avoid attaching identities twice.
    if (
        "person_id" in trades.columns
        or "identity_status" in trades.columns
    ):
        raise ValueError(
            "Trade table already contains identity columns. "
            "Use historical_trades_raw instead."
        )

    # Normalise join keys.
    trades["campaign_id"] = (
        trades["campaign_id"].astype(int)
    )

    trades["account_id"] = normalise_account(
        trades["account_id"]
    )

    if trades["account_id"].isna().any():
        raise ValueError(
            "Trade rows contain missing account IDs."
        )

    # Confirm that each campaign-account has one identity.
    if identity_table.duplicated(join_columns).any():
        raise ValueError(
            "Identity table has duplicate campaign-account keys."
        )

    # Join strictly within campaigns.
    merged = trades.merge(
        identity_table[
            join_columns
            + [
                "person_id",
                "identity_status",
            ]
        ],
        on=join_columns,
        how="left",
        validate="many_to_one",
        sort=False,
    )

    # Ensure the join preserved every trade.
    if len(merged) != len(trades):
        raise AssertionError(
            "Identity join changed the trade row count."
        )

    # Every trading account should now have an identity.
    if merged["person_id"].isna().any():
        unmapped = (
            merged.loc[
                merged["person_id"].isna(),
                join_columns,
            ]
            .drop_duplicates()
        )

        raise ValueError(
            f"{len(unmapped)} campaign-account pairs "
            f"remain unmapped:\n"
            f"{unmapped.head(10).to_string(index=False)}"
        )

    print(
        "Person IDs attached successfully:",
        f"{len(merged):,} trades."
    )

    return merged


# Always join using the original trade data.
historical_trades = attach_person_ids(
    historical_trades_raw,
    historical_users,
)

# ======================================================
# Validate the final trade table
# ======================================================

assert len(historical_trades) == len(
    historical_trades_raw
)

assert historical_trades["person_id"].notna().all()

identity_trade_summary = (
    historical_trades.groupby("identity_status")
    .agg(
        trade_rows=("account_id", "size"),
        unique_person_ids=("person_id", "nunique"),
    )
    .reset_index()
)

print("\nTrade-level identity summary:")
display(identity_trade_summary)

print(
    "\nOriginal trade rows:",
    len(historical_trades_raw),
)

print(
    "Trade rows after identity join:",
    len(historical_trades),
)

print(
    "Missing person IDs:",
    historical_trades["person_id"].isna().sum(),
)

print("\nHistorical identity preparation completed.")
print("Historical identity preparation completed.")


Person IDs attached successfully: 46,520 trades.

Trade-level identity summary:


,identity_status,trade_rows,unique_person_ids
0,email_based,46376,3550
1,missing_email_fallback,116,23
2,missing_user_record_fallback,28,6



Original trade rows: 46520
Trade rows after identity join: 46520
Missing person IDs: 0

Historical identity preparation completed.
Historical identity preparation completed.


# Trade and trade-idea reconstruction


In [16]:
def attach_previous_completed_trade(
    trades,
):
    """Attach the most recent completed trade known at entry time."""

    result_frames = []

    grouping_columns = [
        "campaign_id",
        "account_id",
    ]

    for _, group in (
        trades.groupby(
            grouping_columns,
            dropna=False,
            sort=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "close_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        completed = (
            group.loc[
                group["close_date_time"].notna()
                & group["net_profit"].notna()
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        previous_columns = [
            "close_date_time",
            "net_profit",
            "amount",
        ]

        if (
            "position_id"
            in completed.columns
        ):
            previous_columns.append(
                "position_id"
            )

        previous = (
            completed[
                previous_columns
            ]
            .rename(
                columns={
                    "close_date_time":
                        (
                            "previous_completed_"
                            "close_date_time"
                        ),
                    "net_profit":
                        (
                            "previous_completed_"
                            "net_profit"
                        ),
                    "amount":
                        (
                            "previous_completed_"
                            "amount"
                        ),
                    "position_id":
                        (
                            "previous_completed_"
                            "position_id"
                        ),
                }
            )
        )

        merged = pd.merge_asof(
            current.sort_values(
                "open_date_time"
            ),
            previous.sort_values(
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "previous_completed_"
                "close_date_time"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        result_frames.append(
            merged
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "previous_completed_was_loss"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        < 0
    )

    result[
        "previous_completed_was_win"
    ] = (
        result[
            "previous_completed_net_profit"
        ]
        > 0
    )

    result[
        "reentry_gap_minutes"
    ] = (
        (
            result[
                "open_date_time"
            ]
            - result[
                (
                    "previous_completed_"
                    "close_date_time"
                )
            ]
        )
        .dt.total_seconds()
        / 60
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


In [17]:
def assign_trade_ideas(
    trades,
    maximum_gap_minutes=3.0,
):
    """Assign the Stage 1 directional trade-idea identifier."""

    result = (
        trades.copy()
    )

    group_columns = [
        "account_id",
        "campaign_id",
        "side",
    ]

    result = (
        result
        .sort_values(
            group_columns
            + [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    idea_numbers = pd.Series(
        index=result.index,
        dtype="Int64",
    )

    for _, group in (
        result.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        group = group.sort_values(
            [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )

        current_idea_number = 1
        current_idea_latest_close = (
            pd.NaT
        )

        for row in (
            group.itertuples()
        ):
            if pd.isna(
                current_idea_latest_close
            ):
                current_idea_latest_close = (
                    row.close_date_time
                )

            else:
                gap_minutes = (
                    (
                        row.open_date_time
                        - current_idea_latest_close
                    )
                    .total_seconds()
                    / 60
                )

                if (
                    gap_minutes
                    > maximum_gap_minutes
                ):
                    current_idea_number += 1

                    current_idea_latest_close = (
                        row.close_date_time
                    )

                else:
                    current_idea_latest_close = max(
                        current_idea_latest_close,
                        row.close_date_time,
                    )

            idea_numbers.loc[
                row.Index
            ] = (
                current_idea_number
            )

    result[
        "idea_number"
    ] = (
        idea_numbers
    )

    result[
        "idea_id"
    ] = (
        result[
            "account_id"
        ].astype(str)
        + "_"
        + result[
            "campaign_id"
        ].astype(str)
        + "_"
        + result[
            "side"
        ].astype(str)
        + "_"
        + result[
            "idea_number"
        ].astype(str)
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


In [18]:
def build_stage1_tables_from_trades(trades):
    """Reconstruct trade and idea masters from campaign-joined trade records."""
    trades = trades.copy()

    trades = (
        trades
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "source_row_number",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    trades[
        "trade_row_id"
    ] = np.arange(
        1,
        len(trades) + 1,
    )

    trades_with_previous_completed = (
        attach_previous_completed_trade(
            trades
        )
    )

    trades_with_ideas = (
        assign_trade_ideas(
            trades=trades,
            maximum_gap_minutes=3.0,
        )
    )

    trades_with_ideas = (
        trades_with_ideas
        .sort_values(
            [
                "idea_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .copy()
    )

    trades_with_ideas[
        "trade_number_within_idea"
    ] = (
        trades_with_ideas
        .groupby(
            "idea_id"
        )
        .cumcount()
        + 1
    )

    idea_features = (
        trades_with_ideas
        .groupby(
            "idea_id",
            as_index=False,
        )
        .agg(
            account_id=(
                "account_id",
                "first",
            ),
            person_id=("person_id", "first"),
            outcome_observed=("net_profit", lambda s: s.notna().all()),
            campaign_id=(
                "campaign_id",
                "first",
            ),
            side=(
                "side",
                "first",
            ),
            idea_start_time=(
                "open_date_time",
                "min",
            ),
            idea_end_time=(
                "close_date_time",
                "max",
            ),
            total_amount=(
                "amount",
                "sum",
            ),
            total_net_profit=(
                "net_profit",
                "sum",
            ),
        )
    )

    idea_features[
        "is_profitable_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        > 0
    )

    idea_features[
        "is_losing_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        < 0
    )

    idea_features[
        "is_breakeven_idea"
    ] = (
        idea_features[
            "total_net_profit"
        ]
        == 0
    )

    unseen_trade_row_ids = (
        trades.loc[
            trades[
                "_is_unseen"
            ],
            "trade_row_id",
        ]
        .tolist()
    )

    return {
        "trades":
            trades,
        "trades_with_previous_completed":
            (
                trades_with_previous_completed
            ),
        "trades_with_ideas":
            trades_with_ideas,
        "idea_features":
            idea_features,
        "unseen_trade_row_ids":
            unseen_trade_row_ids,
    }


# Feature engineering


## Pre-trade activity, timing, state and sizing utilities


In [19]:
def attach_past_event_count(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    output_column,
):
    """Attach the number of strictly earlier events."""

    event_counts = (
        event_table
        .groupby(
            group_columns
            + [
                event_time_column
            ],
            dropna=False,
        )
        .size()
        .rename(
            "_events_at_timestamp"
        )
        .reset_index()
    )

    event_counts = (
        event_counts
        .sort_values(
            group_columns
            + [
                event_time_column
            ]
        )
    )

    event_counts[
        output_column
    ] = (
        event_counts
        .groupby(
            group_columns,
            dropna=False,
        )[
            "_events_at_timestamp"
        ]
        .cumsum()
    )

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_counts.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_counts[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_counts[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_counts.loc[
                event_mask,
                [
                    event_time_column,
                    output_column,
                ],
            ]
            .sort_values(
                event_time_column
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                output_column
            ] = 0

            output_groups.append(
                current_group
            )

            continue

        merge_time_column = (
            f"_past_{output_column}_time"
        )

        group_events = (
            group_events.rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = (
            pd.merge_asof(
                left=(
                    current_group
                ),
                right=(
                    group_events
                ),
                left_on=(
                    target_time_column
                ),
                right_on=(
                    merge_time_column
                ),
                direction="backward",
                allow_exact_matches=False,
            )
        )

        current_group[
            output_column
        ] = (
            current_group[
                output_column
            ]
            .fillna(0)
            .astype(int)
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )


In [20]:
def attach_rolling_event_counts(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    window_minutes,
    feature_prefix,
):
    """Attach counts of strictly earlier events in rolling windows."""

    result_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask,
                event_time_column,
            ]
            .dropna()
            .sort_values()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        target_times = (
            current_group[
                target_time_column
            ]
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        event_times = (
            group_events
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        for window in (
            window_minutes
        ):
            left_boundaries = (
                target_times
                - np.timedelta64(
                    window,
                    "m",
                )
            )

            left_indices = (
                np.searchsorted(
                    event_times,
                    left_boundaries,
                    side="left",
                )
            )

            right_indices = (
                np.searchsorted(
                    event_times,
                    target_times,
                    side="left",
                )
            )

            feature_column = (
                f"{feature_prefix}"
                f"_past_{window}_minutes"
            )

            current_group[
                feature_column
            ] = (
                right_indices
                - left_indices
            )

        result_groups.append(
            current_group
        )

    return pd.concat(
        result_groups,
        ignore_index=True,
    )


In [21]:
def attach_entry_spacing_features(
    target_table,
    event_table,
    group_columns,
    target_time_column,
    event_time_column,
    tie_breaker_columns,
    previous_gap_column,
    past_median_gap_column,
    gap_ratio_column,
):
    """Attach pre-entry gap and historical median pace features."""

    output_groups = []

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                event_table.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            event_table.loc[
                event_mask
            ]
            .sort_values(
                [
                    event_time_column
                ]
                + tie_breaker_columns
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                previous_gap_column
            ] = np.nan

            current_group[
                past_median_gap_column
            ] = np.nan

            current_group[
                gap_ratio_column
            ] = np.nan

            output_groups.append(
                current_group
            )

            continue

        event_gap_column = (
            "_entry_gap_minutes"
        )

        group_events[
            event_gap_column
        ] = (
            group_events[
                event_time_column
            ]
            .diff()
            .dt.total_seconds()
            / 60
        )

        group_events[
            past_median_gap_column
        ] = (
            group_events[
                event_gap_column
            ]
            .expanding(
                min_periods=1
            )
            .median()
        )

        merge_time_column = (
            "_previous_event_time"
        )

        events_for_merge = (
            group_events[
                [
                    event_time_column,
                    past_median_gap_column,
                ]
            ]
            .rename(
                columns={
                    event_time_column:
                        merge_time_column
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                events_for_merge
            ),
            left_on=(
                target_time_column
            ),
            right_on=(
                merge_time_column
            ),
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            previous_gap_column
        ] = (
            (
                current_group[
                    target_time_column
                ]
                - current_group[
                    merge_time_column
                ]
            )
            .dt.total_seconds()
            / 60
        )

        valid_ratio = (
            current_group[
                previous_gap_column
            ].ge(0)
            & current_group[
                past_median_gap_column
            ].gt(0)
        )

        current_group[
            gap_ratio_column
        ] = np.where(
            valid_ratio,
            (
                current_group[
                    previous_gap_column
                ]
                / current_group[
                    past_median_gap_column
                ]
            ),
            np.nan,
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )


In [22]:
def attach_realized_challenge_state(
    target_table,
    realized_events,
):
    """Attach cumulative realized P&L known before each trade."""

    result_frames = []

    group_columns = [
        "campaign_id",
        "account_id",
    ]

    for group_key, target_group in (
        target_table.groupby(
            group_columns,
            sort=False,
            dropna=False,
        )
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=(
                realized_events.index
            ),
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(
                value
            ):
                event_mask &= (
                    realized_events[
                        column
                    ].isna()
                )

            else:
                event_mask &= (
                    realized_events[
                        column
                    ].eq(
                        value
                    )
                )

        group_events = (
            realized_events.loc[
                event_mask,
                [
                    "close_date_time",
                    "cumulative_realized_pnl",
                ],
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                "open_date_time"
            )
            .copy()
        )

        if (
            group_events.empty
        ):
            current_group[
                "realized_pnl_before_trade"
            ] = 0.0

            result_frames.append(
                current_group
            )

            continue

        group_events = (
            group_events.rename(
                columns={
                    "close_date_time":
                        (
                            "latest_realized_"
                            "close_before_trade"
                        )
                }
            )
        )

        current_group = pd.merge_asof(
            left=(
                current_group
            ),
            right=(
                group_events
            ),
            left_on=(
                "open_date_time"
            ),
            right_on=(
                "latest_realized_"
                "close_before_trade"
            ),
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            "realized_pnl_before_trade"
        ] = (
            current_group[
                "cumulative_realized_pnl"
            ]
            .fillna(0.0)
        )

        current_group = (
            current_group.drop(
                columns=[
                    "cumulative_realized_pnl"
                ],
                errors="ignore",
            )
        )

        result_frames.append(
            current_group
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )


In [23]:
def attach_historical_position_size_features(
    target_table,
):
    """Attach historical median position-size features."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        medians = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            medians.append(
                np.median(
                    historical_amounts
                )
                if historical_amounts
                else np.nan
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "historical_median_amount_before_trade"
        ] = medians

        result_frames.append(
            current
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "current_to_historical_median_amount_ratio"
    ] = (
        result[
            "amount"
        ]
        / result[
            "historical_median_amount_before_trade"
        ]
    )

    return result


In [24]:
def attach_historical_sizing_consistency_features(
    target_table,
):
    """Attach historical position-size coefficient of variation."""

    result_frames = []

    for _, group in (
        target_table.groupby(
            [
                "campaign_id",
                "account_id",
            ],
            sort=False,
            dropna=False,
        )
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        past_cvs = []

        for row in (
            current.itertuples()
        ):
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )

                same_timestamp_amounts = []

            past_count = len(
                historical_amounts
            )

            past_mean = (
                np.mean(
                    historical_amounts
                )
                if past_count >= 1
                else np.nan
            )

            past_std = (
                np.std(
                    historical_amounts,
                    ddof=0,
                )
                if past_count >= 2
                else np.nan
            )

            past_cv = (
                past_std / past_mean
                if (
                    past_count >= 2
                    and past_mean > 0
                )
                else np.nan
            )

            past_cvs.append(
                past_cv
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "past_amount_cv"
        ] = (
            past_cvs
        )

        result_frames.append(
            current
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )


## Person-keyed completed-idea history


In [25]:
def build_historical_performance_features(
    idea_table,
):
    """Build historical completed-idea performance features."""

    all_ideas = idea_table.copy()
    idea_performance = (
        idea_table.loc[idea_table["outcome_observed"].eq(True)
                       & idea_table["idea_end_time"].notna()][
            [
                "idea_id",
                "person_id",
                "campaign_id",
                "idea_start_time",
                "idea_end_time",
                "total_amount",
                "total_net_profit",
                "is_profitable_idea",
                "is_losing_idea",
                "is_breakeven_idea",
            ]
        ]
        .copy()
    )

    idea_performance[
        "idea_profit_per_lot"
    ] = (
        idea_performance[
            "total_net_profit"
        ]
        / idea_performance[
            "total_amount"
        ]
    )

    idea_performance[
        "_win_count"
    ] = (
        idea_performance[
            "is_profitable_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_loss_count"
    ] = (
        idea_performance[
            "is_losing_idea"
        ]
        .astype(int)
    )

    idea_performance[
        "_idea_count"
    ] = 1

    idea_performance[
        "_win_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .where(
            idea_performance[
                "is_profitable_idea"
            ],
            0.0,
        )
    )

    idea_performance[
        "_loss_abs_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .abs()
        .where(
            idea_performance[
                "is_losing_idea"
            ],
            0.0,
        )
    )

    if idea_performance.empty:
        current_ideas = all_ideas[["idea_id", "person_id", "campaign_id", "idea_start_time"]].copy()
        for name in ["past_completed_idea_count", "past_idea_win_rate", "past_mean_win_profit_per_lot",
                     "past_mean_loss_abs_profit_per_lot", "past_payoff_ratio"]:
            current_ideas[name] = np.nan
        return current_ideas

    events = (
        idea_performance
        .groupby(
            [
                "person_id",
                "idea_end_time",
            ],
            as_index=False,
        )
        .agg(
            completed_idea_count=(
                "_idea_count",
                "sum",
            ),
            completed_win_count=(
                "_win_count",
                "sum",
            ),
            completed_loss_count=(
                "_loss_count",
                "sum",
            ),
            completed_win_profit_sum=(
                "_win_profit",
                "sum",
            ),
            completed_loss_abs_profit_sum=(
                "_loss_abs_profit",
                "sum",
            ),
        )
        .sort_values(
            [
                "person_id",
                "idea_end_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    cumulative_mapping = {
        "past_completed_idea_count":
            "completed_idea_count",
        "past_winning_idea_count":
            "completed_win_count",
        "past_losing_idea_count":
            "completed_loss_count",
        "past_win_profit_sum":
            "completed_win_profit_sum",
        "past_loss_abs_profit_sum":
            (
                "completed_loss_"
                "abs_profit_sum"
            ),
    }

    for (
        output_column,
        source_column,
    ) in (
        cumulative_mapping.items()
    ):
        events[
            output_column
        ] = (
            events
            .groupby(
                "person_id"
            )[
                source_column
            ]
            .cumsum()
        )

    current_ideas = (
        all_ideas[
            [
                "idea_id",
                "person_id",
                "campaign_id",
                "idea_start_time",
            ]
        ]
        .sort_values(
            [
                "idea_start_time",
                "person_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    events = (
        events
        .sort_values(
            [
                "idea_end_time",
                "person_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    historical = pd.merge_asof(
        current_ideas,
        events[
            [
                "person_id",
                "idea_end_time",
                "past_completed_idea_count",
                "past_winning_idea_count",
                "past_losing_idea_count",
                "past_win_profit_sum",
                "past_loss_abs_profit_sum",
            ]
        ],
        left_on=(
            "idea_start_time"
        ),
        right_on=(
            "idea_end_time"
        ),
        by="person_id",
        direction="backward",
        allow_exact_matches=False,
    )

    historical[
        "past_idea_win_rate"
    ] = (
        historical[
            "past_winning_idea_count"
        ]
        / historical[
            "past_completed_idea_count"
        ]
    )

    historical[
        "past_mean_win_profit_per_lot"
    ] = (
        historical[
            "past_win_profit_sum"
        ]
        / historical[
            "past_winning_idea_count"
        ]
    )

    historical[
        "past_mean_loss_abs_profit_per_lot"
    ] = (
        historical[
            "past_loss_abs_profit_sum"
        ]
        / historical[
            "past_losing_idea_count"
        ]
    )

    historical[
        "past_payoff_ratio"
    ] = (
        historical[
            "past_mean_win_profit_per_lot"
        ]
        / historical[
            "past_mean_loss_abs_profit_per_lot"
        ]
    )

    return historical


## Build the Final Model Feature Table

In [26]:
def build_model_features(
    stage1_tables,
):
    """Recreate the 22 pre-trade features used by the frozen model."""

    trades = (
        stage1_tables[
            "trades"
        ]
        .copy()
    )

    trades_with_ideas = (
        stage1_tables[
            "trades_with_ideas"
        ]
        .copy()
    )

    previous_completed = (
        stage1_tables[
            "trades_with_previous_completed"
        ]
        .copy()
    )

    idea_features = (
        stage1_tables[
            "idea_features"
        ]
        .copy()
    )

    trade_features = (
        trades_with_ideas
        .copy()
    )

    # --------------------------------------------------------
    # Loss response
    # --------------------------------------------------------

    previous_columns = [
        "trade_row_id",
        "previous_completed_close_date_time",
        "previous_completed_net_profit",
        "previous_completed_amount",
        "previous_completed_was_loss",
        "previous_completed_was_win",
        "reentry_gap_minutes",
    ]

    trade_features = (
        trade_features.merge(
            previous_completed[
                previous_columns
            ],
            on="trade_row_id",
            how="left",
            validate="one_to_one",
        )
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "amount"
        ]
        / trade_features[
            "previous_completed_amount"
        ]
    )

    trade_features[
        "current_to_previous_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    trade_features[
        "post_loss_entry"
    ] = (
        trade_features[
            "previous_completed_was_loss"
        ]
        .fillna(False)
        .astype(bool)
    )

    trade_features[
        "post_loss_reentry_gap_minutes"
    ] = (
        trade_features[
            "reentry_gap_minutes"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    trade_features[
        "post_loss_amount_ratio"
    ] = (
        trade_features[
            "current_to_previous_amount_ratio"
        ]
        .where(
            trade_features[
                "post_loss_entry"
            ]
        )
    )

    # --------------------------------------------------------
    # Activity
    # --------------------------------------------------------

    trade_features = (
        attach_past_event_count(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            output_column=(
                "past_trade_open_count"
            ),
        )
    )

    trade_features[
        "first_trade_open_date_time"
    ] = (
        trade_features
        .groupby(
            [
                "account_id",
                "campaign_id",
            ]
        )[
            "open_date_time"
        ]
        .transform(
            "min"
        )
    )

    trade_features[
        "elapsed_active_hours"
    ] = (
        (
            trade_features[
                "open_date_time"
            ]
            - trade_features[
                "first_trade_open_date_time"
            ]
        )
        .dt.total_seconds()
        / 3600
    )

    valid_elapsed = (
        trade_features[
            "elapsed_active_hours"
        ]
        > 0
    )

    trade_features[
        "past_trades_opened_per_active_hour"
    ] = np.where(
        valid_elapsed,
        (
            trade_features[
                "past_trade_open_count"
            ]
            / trade_features[
                "elapsed_active_hours"
            ]
        ),
        np.nan,
    )

    trade_features = (
        attach_rolling_event_counts(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            window_minutes=[
                30,
            ],
            feature_prefix=(
                "trades_opened"
            ),
        )
    )

    # --------------------------------------------------------
    # Trade timing
    # --------------------------------------------------------

    trade_features = (
        attach_entry_spacing_features(
            target_table=(
                trade_features
            ),
            event_table=trades,
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "open_date_time"
            ),
            event_time_column=(
                "open_date_time"
            ),
            tie_breaker_columns=[
                "trade_row_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_trade_open"
            ),
            past_median_gap_column=(
                "past_median_trade_open_gap_minutes"
            ),
            gap_ratio_column=(
                "trade_open_gap_to_past_median_ratio"
            ),
        )
    )

    idea_start_events = (
        idea_features[
            [
                "account_id",
                "campaign_id",
                "idea_id",
                "idea_start_time",
            ]
        ]
        .drop_duplicates(
            subset=[
                "idea_id"
            ]
        )
        .copy()
    )

    idea_features = (
        attach_entry_spacing_features(
            target_table=(
                idea_features
            ),
            event_table=(
                idea_start_events
            ),
            group_columns=[
                "account_id",
                "campaign_id",
            ],
            target_time_column=(
                "idea_start_time"
            ),
            event_time_column=(
                "idea_start_time"
            ),
            tie_breaker_columns=[
                "idea_id",
            ],
            previous_gap_column=(
                "minutes_since_previous_idea_start"
            ),
            past_median_gap_column=(
                "past_median_idea_start_gap_minutes"
            ),
            gap_ratio_column=(
                "idea_start_gap_to_past_median_ratio"
            ),
        )
    )

    trade_features = (
        trade_features.merge(
            idea_features[
                [
                    "idea_id",
                    "minutes_since_previous_idea_start",
                    "past_median_idea_start_gap_minutes",
                    "idea_start_gap_to_past_median_ratio",
                ]
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # --------------------------------------------------------
    # Challenge state
    # --------------------------------------------------------

    realized_events = (
        trades[
            [
                "account_id",
                "campaign_id",
                "close_date_time",
                "net_profit",
            ]
        ]
        .dropna(
            subset=[
                "close_date_time",
                "net_profit",
            ]
        )
        .groupby(
            [
                "account_id",
                "campaign_id",
                "close_date_time",
            ],
            as_index=False,
        )[
            "net_profit"
        ]
        .sum()
        .rename(
            columns={
                "net_profit":
                    "realized_pnl_at_close"
            }
        )
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "close_date_time",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    realized_events[
        "cumulative_realized_pnl"
    ] = (
        realized_events
        .groupby(
            [
                "campaign_id",
                "account_id",
            ]
        )[
            "realized_pnl_at_close"
        ]
        .cumsum()
    )

    trade_features = (
        attach_realized_challenge_state(
            target_table=(
                trade_features
            ),
            realized_events=(
                realized_events
            ),
        )
    )

    trade_features[
        "realized_distance_to_drawdown_limit"
    ] = (
        trade_features[
            "realized_pnl_before_trade"
        ]
        - REALIZED_DRAWDOWN_BOUNDARY
    )

    # --------------------------------------------------------
    # Position sizing
    # --------------------------------------------------------

    trade_features = (
        attach_historical_position_size_features(
            trade_features
        )
    )

    trade_features = (
        attach_historical_sizing_consistency_features(
            trade_features
        )
    )

    # --------------------------------------------------------
    # Historical idea performance
    # --------------------------------------------------------

    historical_idea_features = (
        build_historical_performance_features(
            idea_features
        )
    )

    historical_columns = [
        "idea_id",
        "past_completed_idea_count",
        "past_idea_win_rate",
        "past_mean_win_profit_per_lot",
        "past_mean_loss_abs_profit_per_lot",
        "past_payoff_ratio",
    ]

    trade_features = (
        trade_features.merge(
            historical_idea_features[
                historical_columns
            ],
            on="idea_id",
            how="left",
            validate="many_to_one",
        )
    )

    # Replace invalid numerical values exactly as missing values.
    trade_features = (
        trade_features.replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
    )

    if (
        "reverse_profit"
        in trade_features.columns
        and "amount"
        in trade_features.columns
    ):
        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit"
            ]
            / trade_features[
                "amount"
            ]
        )

        trade_features[
            "reverse_profit_per_lot"
        ] = (
            trade_features[
                "reverse_profit_per_lot"
            ]
            .replace(
                [
                    np.inf,
                    -np.inf,
                ],
                np.nan,
            )
        )

    return (
        trade_features
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# Build train and validation datasets


In [27]:
def build_campaign_deadlines(trades):
    """Build the confirmed 9 AM-to-9 AM SGT deadline per campaign."""
    campaign_dates = trades[
        ["campaign_id", "campaign_date"]
    ].drop_duplicates()

    if (
        campaign_dates["campaign_date"].isna().any()
        or campaign_dates["campaign_id"].duplicated().any()
    ):
        raise ValueError(
            "Missing or conflicting campaign start dates."
        )

    return {
        int(row.campaign_id): (
            pd.Timestamp(row.campaign_date)
            .normalize()
            .tz_localize(CHALLENGE_TIMEZONE)
            + pd.Timedelta(
                hours=(
                    CHALLENGE_START_HOUR
                    + CHALLENGE_DURATION_HOURS
                )
            )
        ).tz_convert("UTC")
        for row in campaign_dates.itertuples(index=False)
    }


expected_historical_campaigns = set(
    TRAIN_CAMPAIGNS + VALIDATION_CAMPAIGNS
)

actual_historical_campaigns = set(
    historical_trades["campaign_id"].astype(int).unique()
)

if actual_historical_campaigns != expected_historical_campaigns:
    raise ValueError(
        "Historical campaigns do not match campaigns 33–66."
    )

if historical_trades["person_id"].isna().any():
    raise ValueError(
        "Historical trades contain missing person IDs."
    )

historical_trades = historical_trades.copy()
historical_trades["_is_unseen"] = False

stage1_historical = build_stage1_tables_from_trades(
    historical_trades
)

historical_features = build_model_features(
    stage1_historical
)

if len(historical_features) != len(historical_trades):
    raise AssertionError(
        "Feature engineering changed the historical trade-row count."
    )

missing_feature_columns = (
    set(FEATURE_COLUMNS) - set(historical_features.columns)
)

if missing_feature_columns:
    raise AssertionError(
        "Historical feature table is missing frozen features: "
        f"{sorted(missing_feature_columns)}"
    )

if historical_features["reverse_profit_per_lot"].isna().any():
    raise ValueError(
        "Historical features contain missing training outcomes."
    )

if historical_features["trade_row_id"].duplicated().any():
    raise AssertionError(
        "Historical feature table contains duplicate trade_row_id values."
    )

CAMPAIGN_DEADLINES_UTC = build_campaign_deadlines(
    historical_trades
)

print(
    f"Prepared {len(historical_features):,} historical rows "
    f"with {len(FEATURE_COLUMNS)} frozen features."
)


Prepared 46,520 historical rows with 22 frozen features.


# Define evaluations metrics


In [28]:
# ======================================================
# Metrics and Holm correction
# ======================================================


def lot_weighted_edge(data):
    """
    Calculate total reverse profit divided by total lots.

    This is the required lot-weighted reverse profit per lot,
    rather than the unweighted mean of trade-level RP/lot.
    """
    if data.empty:
        return np.nan

    total_lots = data["amount"].sum()

    if (
        not np.isfinite(total_lots)
        or total_lots <= 0
    ):
        return np.nan

    return float(
        data["reverse_profit"].sum()
        / total_lots
    )


def person_cluster_bootstrap(
    data,
    n_bootstrap=BOOTSTRAP_ITERATIONS,
    seed=SEED,
):
    """
    Bootstrap lot-weighted RP/lot by resampling entire people.

    All selected trades belonging to one person remain together
    within a bootstrap sample.
    """
    if data.empty:
        return np.array(
            [],
            dtype=float,
        )

    required_columns = {
        "person_id",
        "reverse_profit",
        "amount",
    }

    missing_columns = (
        required_columns
        - set(data.columns)
    )

    if missing_columns:
        raise ValueError(
            "Bootstrap data is missing columns: "
            f"{sorted(missing_columns)}"
        )

    if data[
        list(required_columns)
    ].isna().any().any():
        raise ValueError(
            "Bootstrap data contains missing person IDs, "
            "reverse-profit values, or trade amounts."
        )

    person_totals = (
        data.groupby(
            "person_id",
            sort=True,
        )
        .agg(
            reverse_profit=(
                "reverse_profit",
                "sum",
            ),
            amount=(
                "amount",
                "sum",
            ),
        )
        .reset_index(drop=True)
    )

    if person_totals.empty:
        return np.array(
            [],
            dtype=float,
        )

    if person_totals["amount"].le(0).any():
        raise ValueError(
            "Person-level bootstrap totals contain "
            "non-positive trade amounts."
        )

    values = person_totals[
        [
            "reverse_profit",
            "amount",
        ]
    ].to_numpy(
        dtype=float
    )

    number_of_people = len(values)

    rng = np.random.default_rng(
        seed
    )

    sampled_indices = rng.integers(
        low=0,
        high=number_of_people,
        size=(
            n_bootstrap,
            number_of_people,
        ),
    )

    sampled_totals = (
        values[
            sampled_indices
        ]
        .sum(axis=1)
    )

    samples = (
        sampled_totals[:, 0]
        / sampled_totals[:, 1]
    )

    return samples


def percentile_bootstrap_ci(
    samples,
    confidence=CONFIDENCE_LEVEL,
):
    """
    Calculate a two-sided percentile bootstrap interval.
    """
    samples = np.asarray(
        samples,
        dtype=float,
    )

    samples = samples[
        np.isfinite(samples)
    ]

    if len(samples) == 0:
        return np.nan, np.nan

    alpha = 1.0 - confidence

    lower, upper = np.quantile(
        samples,
        [
            alpha / 2.0,
            1.0 - alpha / 2.0,
        ],
    )

    return float(lower), float(upper)


def centered_bootstrap_positive_pvalue(
    observed,
    samples,
):
    """
    Calculate a one-sided, null-centered bootstrap p-value.

    Null hypothesis:
        lot-weighted RP/lot <= 0

    Alternative hypothesis:
        lot-weighted RP/lot > 0
    """
    samples = np.asarray(
        samples,
        dtype=float,
    )

    samples = samples[
        np.isfinite(samples)
    ]

    if (
        len(samples) == 0
        or not np.isfinite(observed)
    ):
        return np.nan

    # The ordinary bootstrap distribution is centered around
    # the observed edge. Subtracting the observed edge creates
    # a null distribution centered around zero.
    null_samples = (
        samples
        - observed
    )

    # Add one to the numerator and denominator to prevent a
    # simulated p-value of exactly zero.
    extreme_count = np.count_nonzero(
        null_samples >= observed
    )

    p_value = (
        extreme_count + 1
    ) / (
        len(null_samples) + 1
    )

    return float(p_value)


def holm_adjust(p_values):
    """
    Return step-down Holm-adjusted p-values for all
    enabled framework tests.
    """
    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    if len(p_values) == 0:
        return np.array(
            [],
            dtype=float,
        )

    if not np.isfinite(
        p_values
    ).all():
        return np.full(
            len(p_values),
            np.nan,
            dtype=float,
        )

    if (
        (p_values < 0).any()
        or (p_values > 1).any()
    ):
        raise ValueError(
            "All p-values must lie between zero and one."
        )

    number_of_tests = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty(
        number_of_tests,
        dtype=float,
    )

    running_maximum = 0.0

    for rank, original_index in enumerate(
        order
    ):
        remaining_tests = (
            number_of_tests
            - rank
        )

        candidate = min(
            1.0,
            remaining_tests
            * p_values[original_index],
        )

        running_maximum = max(
            running_maximum,
            candidate,
        )

        adjusted[
            original_index
        ] = running_maximum

    return adjusted


def evaluate_frameworks(
    decisions,
):
    """
    Evaluate coverage, lot-weighted edge, campaign stability,
    person-cluster uncertainty, and registered hypothesis tests.
    """
    required_columns = {
        "trade_row_id",
        "campaign_id",
        "person_id",
        "amount",
        "reverse_profit",
        "framework",
        "threshold",
        "fade_decision",
    }

    missing_columns = (
        required_columns
        - set(decisions.columns)
    )

    if missing_columns:
        raise ValueError(
            "Decision table is missing columns: "
            f"{sorted(missing_columns)}"
        )

    summaries = []
    campaign_frames = []
    bootstrap_rows = []

    outcomes_available = (
        decisions[
            [
                "reverse_profit",
                "amount",
            ]
        ]
        .notna()
        .all()
        .all()
    )

    for framework_name, framework_data in decisions.groupby(
        "framework",
        sort=False,
    ):
        selected = (
            framework_data.loc[
                framework_data[
                    "fade_decision"
                ]
            ]
            .copy()
        )

        trades_evaluated = len(
            framework_data
        )

        trades_faded = len(
            selected
        )

        coverage = (
            trades_faded
            / trades_evaluated
            if trades_evaluated
            else np.nan
        )

        summary_row = {
            "framework": framework_name,
            "trades_evaluated": trades_evaluated,
            "trades_faded": trades_faded,
            "coverage": coverage,
        }

        if not outcomes_available:
            summary_row.update(
                {
                    "lot_weighted_rp_per_lot": np.nan,
                    "ci_lower": np.nan,
                    "ci_upper": np.nan,
                    "positive_campaigns": np.nan,
                    "total_campaigns": np.nan,
                    "positive_campaign_share": np.nan,
                    "p_value_one_sided": np.nan,
                }
            )

            summaries.append(
                summary_row
            )

            continue

        observed_edge = lot_weighted_edge(
            selected
        )

        bootstrap_samples = (
            person_cluster_bootstrap(
                selected,
                n_bootstrap=BOOTSTRAP_ITERATIONS,
                seed=SEED,
            )
        )

        ci_lower, ci_upper = (
            percentile_bootstrap_ci(
                bootstrap_samples,
                confidence=CONFIDENCE_LEVEL,
            )
        )

        p_value = (
            centered_bootstrap_positive_pvalue(
                observed=observed_edge,
                samples=bootstrap_samples,
            )
        )

        campaign_rows = []

        for campaign_id, campaign_data in (
            framework_data.groupby(
                "campaign_id",
                sort=True,
            )
        ):
            faded = (
                campaign_data.loc[
                    campaign_data[
                        "fade_decision"
                    ]
                ]
                .copy()
            )

            campaign_thresholds = (
                campaign_data[
                    "threshold"
                ]
                .dropna()
                .unique()
            )

            if len(campaign_thresholds) != 1:
                raise AssertionError(
                    f"{framework_name}, campaign {campaign_id}: "
                    "expected exactly one decision threshold."
                )

            campaign_rows.append(
                {
                    "framework": framework_name,
                    "campaign_id": campaign_id,
                    "trades_evaluated": len(
                        campaign_data
                    ),
                    "trades_faded": len(
                        faded
                    ),
                    "coverage": (
                        len(faded)
                        / len(campaign_data)
                    ),
                    "lot_weighted_rp_per_lot": (
                        lot_weighted_edge(
                            faded
                        )
                    ),
                    "decision_threshold": float(
                        campaign_thresholds[0]
                    ),
                }
            )

        campaign_frame = pd.DataFrame(
            campaign_rows
        )

        campaign_frames.append(
            campaign_frame
        )

        positive_campaigns = int(
            campaign_frame[
                "lot_weighted_rp_per_lot"
            ]
            .gt(0)
            .sum()
        )

        total_campaigns = len(
            campaign_frame
        )

        positive_campaign_share = (
            positive_campaigns
            / total_campaigns
            if total_campaigns
            else np.nan
        )

        summary_row.update(
            {
                "lot_weighted_rp_per_lot": observed_edge,
                "ci_lower": ci_lower,
                "ci_upper": ci_upper,
                "positive_campaigns": positive_campaigns,
                "total_campaigns": total_campaigns,
                "positive_campaign_share": (
                    positive_campaign_share
                ),
                "p_value_one_sided": p_value,
            }
        )

        summaries.append(
            summary_row
        )

        bootstrap_rows.extend(
            {
                "framework": framework_name,
                "iteration": iteration + 1,
                "lot_weighted_rp_per_lot": value,
            }
            for iteration, value
            in enumerate(
                bootstrap_samples
            )
        )

        print(
            f"  {framework_name}: "
            f"faded={trades_faded:,}, "
            f"coverage={coverage:.2%}, "
            f"edge={observed_edge:.4f}, "
            f"{CONFIDENCE_LEVEL:.0%} CI="
            f"[{ci_lower:.4f}, {ci_upper:.4f}], "
            f"positive campaigns="
            f"{positive_campaigns}/{total_campaigns}"
        )

    evaluation_summary = pd.DataFrame(
        summaries
    )

    if outcomes_available:
        expected_framework_count = decisions[
            "framework"
        ].nunique()

        if len(evaluation_summary) != expected_framework_count:
            raise AssertionError(
                "The number of evaluated strategies does not "
                "match the decision-table framework count."
            )

        evaluation_summary[
            "holm_adjusted_p"
        ] = holm_adjust(
            evaluation_summary[
                "p_value_one_sided"
            ].to_numpy()
        )

        evaluation_summary[
            "holm_significant"
        ] = (
            evaluation_summary[
                "holm_adjusted_p"
            ]
            < 0.05
        )

        evaluation_summary[
            "edge_above_zero"
        ] = (
            evaluation_summary[
                "lot_weighted_rp_per_lot"
            ]
            > 0
        )

        evaluation_summary[
            "ci_clears_zero"
        ] = (
            evaluation_summary[
                "ci_lower"
            ]
            > 0
        )

        evaluation_summary[
            "campaign_share_pass"
        ] = (
            evaluation_summary[
                "positive_campaign_share"
            ]
            >= MIN_POSITIVE_CAMPAIGN_SHARE
        )

        evaluation_summary[
            "passes_required_criteria"
        ] = (
            evaluation_summary[
                "edge_above_zero"
            ]
            & evaluation_summary[
                "ci_clears_zero"
            ]
            & evaluation_summary[
                "campaign_share_pass"
            ]
        )

    else:
        evaluation_summary[
            "holm_adjusted_p"
        ] = np.nan

        evaluation_summary[
            "holm_significant"
        ] = False

        evaluation_summary[
            "edge_above_zero"
        ] = False

        evaluation_summary[
            "ci_clears_zero"
        ] = False

        evaluation_summary[
            "campaign_share_pass"
        ] = False

        evaluation_summary[
            "passes_required_criteria"
        ] = False

    campaign_output = (
        pd.concat(
            campaign_frames,
            ignore_index=True,
        )
        if campaign_frames
        else pd.DataFrame()
    )

    bootstrap_output = pd.DataFrame(
        bootstrap_rows
    )

    return (
        evaluation_summary,
        campaign_output,
        bootstrap_output,
        outcomes_available,
    )


# Shared model and decision-table utilities


In [29]:
BANNED_MODEL_COLUMNS = {
    "profit",
    "net_profit",
    "reverse_profit",
    "reverse_profit_per_lot",
    "sl_price",
    "tp_price",
    "person_id",
    "account_id",
    "email",
    "ip_address",
}

if (
    len(FEATURE_COLUMNS) != 22
    or BANNED_MODEL_COLUMNS.intersection(FEATURE_COLUMNS)
):
    raise ValueError(
        "The frozen model feature specification is invalid."
    )


def new_regressor():
    """Instantiate the frozen LightGBM model specification."""
    return LGBMRegressor(**MODEL_PARAMETERS)


def fit_regressor(data):
    """Fit the frozen model using complete historical outcomes."""
    if data.empty:
        raise ValueError("Model training data is empty.")

    if data["reverse_profit_per_lot"].isna().any():
        raise ValueError(
            "Model training requires complete historical outcomes."
        )

    model = new_regressor()

    model.fit(
        data[FEATURE_COLUMNS],
        data["reverse_profit_per_lot"],
    )

    return model


def calibrate_training_threshold(model, training_data):
    """Calculate the registered q90 threshold from training predictions."""
    training_scores = np.asarray(
        model.predict(training_data[FEATURE_COLUMNS]),
        dtype=float,
    )

    if not np.isfinite(training_scores).all():
        raise ValueError(
            "Model produced non-finite training scores."
        )

    return float(
        np.quantile(training_scores, QUANTILE)
    )


def build_decision_frame(
    current,
    framework_name,
    scores,
    thresholds,
    fade_decisions,
):
    """Return the common trade-level output schema for one framework."""
    frame = current[
        [
            "trade_row_id",
            "campaign_id",
            "account_id",
            "person_id",
            "amount",
            "reverse_profit",
            "open_date_time",
        ]
    ].copy()

    frame["framework"] = framework_name
    frame["score"] = np.asarray(scores, dtype=float)
    frame["threshold"] = np.asarray(thresholds, dtype=float)
    frame["fade_decision"] = np.asarray(
        fade_decisions,
        dtype=bool,
    )

    if frame["trade_row_id"].duplicated().any():
        raise AssertionError(
            f"{framework_name} produced duplicate trade decisions."
        )

    return frame


# Framework 1 — walk-forward baseline


In [30]:
def run_walk_forward_baseline(
    feature_data,
    evaluation_campaigns,
    framework_name="walk_forward_baseline",
):
    """
    Refit on all earlier campaigns, calibrate a q90 training-score
    threshold, and score the current campaign.
    """
    decision_frames = []

    for campaign_id in evaluation_campaigns:
        training_data = feature_data.loc[
            feature_data["campaign_id"] < campaign_id
        ].copy()

        current = feature_data.loc[
            feature_data["campaign_id"] == campaign_id
        ].copy()

        if training_data.empty or current.empty:
            raise ValueError(
                f"No train/test data for campaign {campaign_id}."
            )

        model = fit_regressor(training_data)

        threshold = calibrate_training_threshold(
            model,
            training_data,
        )

        scores = np.asarray(
            model.predict(current[FEATURE_COLUMNS]),
            dtype=float,
        )

        decisions = scores >= threshold

        decision_frames.append(
            build_decision_frame(
                current=current,
                framework_name=framework_name,
                scores=scores,
                thresholds=np.full(len(current), threshold),
                fade_decisions=decisions,
            )
        )

        print(
            f"{framework_name}, campaign {campaign_id}: "
            f"trained={len(training_data):,}, "
            f"scored={len(current):,}, "
            f"threshold={threshold:.6f}, "
            f"coverage={decisions.mean():.2%}"
        )

    return pd.concat(
        decision_frames,
        ignore_index=True,
    )


## Shared experiment utilities

In [63]:
# ======================================================
# Framework 1 experiments: Shared utilities
# ======================================================

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import (
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor,
)


FRAMEWORK_1_FULL_FEATURES = [
    # Loss-response features
    "post_loss_entry",
    "post_loss_reentry_gap_minutes",
    "post_loss_amount_ratio",
    "post_loss_amount_increased",

    # Trading-activity and timing features
    "past_trade_open_count",
    "past_idea_start_count",
    "elapsed_active_minutes",
    "elapsed_active_hours",
    "past_trades_opened_per_active_hour",
    "past_ideas_started_per_active_hour",
    "trades_opened_past_15_minutes",
    "trades_opened_past_30_minutes",
    "trades_opened_past_60_minutes",
    "ideas_started_past_15_minutes",
    "ideas_started_past_30_minutes",
    "ideas_started_past_60_minutes",
    "minutes_since_previous_trade_open",
    "past_median_trade_open_gap_minutes",
    "trade_open_gap_to_past_median_ratio",
    "minutes_since_previous_idea_start",
    "past_median_idea_start_gap_minutes",
    "idea_start_gap_to_past_median_ratio",

    # Challenge-state features
    "realized_pnl_before_trade",
    "realized_return_before_trade",
    "realized_balance_before_trade",
    "realized_distance_to_drawdown_limit",
    "realized_distance_to_profit_target",

    # Position-sizing features
    "historical_median_amount_before_trade",
    "current_to_historical_median_amount_ratio",
    "past_amount_count",
    "past_amount_mean",
    "past_amount_std",
    "past_amount_cv",

    # Historical-performance features
    "past_completed_idea_count",
    "past_winning_idea_count",
    "past_losing_idea_count",
    "past_idea_win_rate",
    "past_mean_win_profit_per_lot",
    "past_mean_loss_abs_profit_per_lot",
    "past_payoff_ratio",
]


FRAMEWORK_1_BANNED_COLUMNS = {
    "profit",
    "net_profit",
    "reverse_profit",
    "reverse_profit_per_lot",
    "sl_price",
    "tp_price",
    "person_id",
    "account_id",
    "email",
    "ip_address",
}


if len(FEATURE_COLUMNS) != 22:
    raise ValueError(
        "FEATURE_COLUMNS is expected to contain the "
        "reduced 22-feature specification."
    )

if len(FRAMEWORK_1_FULL_FEATURES) != 40:
    raise ValueError(
        "FRAMEWORK_1_FULL_FEATURES must contain exactly "
        "40 features."
    )

if FRAMEWORK_1_BANNED_COLUMNS.intersection(
    FRAMEWORK_1_FULL_FEATURES
):
    raise ValueError(
        "The full feature set contains banned or "
        "outcome-leaking columns."
    )


def make_framework_1_estimator(
    model_name,
    model_parameters=None,
):
    """Construct one candidate regression model."""

    model_parameters = (
        {}
        if model_parameters is None
        else model_parameters.copy()
    )

    if model_name == "lightgbm":
        default_parameters = {
            "n_estimators": 200,
            "learning_rate": 0.05,
            "num_leaves": 15,
            "random_state": SEED,
            "verbosity": -1,
        }

        default_parameters.update(
            model_parameters
        )

        return LGBMRegressor(
            **default_parameters
        )

    if model_name == "ridge":
        default_parameters = {
            "alpha": 10.0,
        }

        default_parameters.update(
            model_parameters
        )

        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "model",
                    Ridge(
                        **default_parameters
                    ),
                ),
            ]
        )

    if model_name == "elastic_net":
        default_parameters = {
            "alpha": 0.1,
            "l1_ratio": 0.5,
            "max_iter": 10_000,
            "random_state": SEED,
        }

        default_parameters.update(
            model_parameters
        )

        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "model",
                    ElasticNet(
                        **default_parameters
                    ),
                ),
            ]
        )

    if model_name == "extra_trees":
        default_parameters = {
            "n_estimators": 300,
            "min_samples_leaf": 10,
            "max_features": 0.8,
            "n_jobs": -1,
            "random_state": SEED,
        }

        default_parameters.update(
            model_parameters
        )

        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "model",
                    ExtraTreesRegressor(
                        **default_parameters
                    ),
                ),
            ]
        )

    if model_name == "random_forest":
        default_parameters = {
            "n_estimators": 300,
            "min_samples_leaf": 10,
            "max_features": 0.8,
            "n_jobs": -1,
            "random_state": SEED,
        }

        default_parameters.update(
            model_parameters
        )

        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "model",
                    RandomForestRegressor(
                        **default_parameters
                    ),
                ),
            ]
        )

    if model_name == "hist_gradient_boosting":
        default_parameters = {
            "learning_rate": 0.05,
            "max_iter": 200,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 20,
            "l2_regularization": 1.0,
            "random_state": SEED,
        }

        default_parameters.update(
            model_parameters
        )

        return Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median"
                    ),
                ),
                (
                    "model",
                    HistGradientBoostingRegressor(
                        **default_parameters
                    ),
                ),
            ]
        )

    raise ValueError(
        f"Unknown model_name: {model_name}"
    )


def run_framework_1_experiment(
    feature_data,
    evaluation_campaigns,
    framework_name,
    feature_columns,
    model_name="lightgbm",
    model_parameters=None,
    selection_quantile=0.90,
):
    """
    Run an expanding-window walk-forward experiment.

    For evaluation campaign c, the model is trained only on
    campaigns strictly earlier than c.
    """

    if not 0 < selection_quantile < 1:
        raise ValueError(
            "selection_quantile must lie between zero and one."
        )

    missing_features = (
        set(feature_columns)
        - set(feature_data.columns)
    )

    if missing_features:
        raise ValueError(
            f"{framework_name} is missing features: "
            f"{sorted(missing_features)}"
        )

    banned_features = (
        set(feature_columns)
        & FRAMEWORK_1_BANNED_COLUMNS
    )

    if banned_features:
        raise ValueError(
            f"{framework_name} contains banned features: "
            f"{sorted(banned_features)}"
        )

    decision_frames = []

    for campaign_id in evaluation_campaigns:
        training_data = (
            feature_data.loc[
                feature_data[
                    "campaign_id"
                ] < campaign_id
            ]
            .copy()
        )

        current_data = (
            feature_data.loc[
                feature_data[
                    "campaign_id"
                ] == campaign_id
            ]
            .copy()
        )

        if training_data.empty:
            raise ValueError(
                f"{framework_name}, campaign {campaign_id}: "
                "training data is empty."
            )

        if current_data.empty:
            raise ValueError(
                f"{framework_name}, campaign {campaign_id}: "
                "evaluation data is empty."
            )

        if training_data[
            "reverse_profit_per_lot"
        ].isna().any():
            raise ValueError(
                f"{framework_name}, campaign {campaign_id}: "
                "training outcomes contain missing values."
            )

        estimator = make_framework_1_estimator(
            model_name=model_name,
            model_parameters=model_parameters,
        )

        estimator.fit(
            training_data[
                feature_columns
            ],
            training_data[
                "reverse_profit_per_lot"
            ],
        )

        training_scores = np.asarray(
            estimator.predict(
                training_data[
                    feature_columns
                ]
            ),
            dtype=float,
        )

        current_scores = np.asarray(
            estimator.predict(
                current_data[
                    feature_columns
                ]
            ),
            dtype=float,
        )

        if not np.isfinite(
            training_scores
        ).all():
            raise ValueError(
                f"{framework_name}, campaign {campaign_id}: "
                "non-finite training scores."
            )

        if not np.isfinite(
            current_scores
        ).all():
            raise ValueError(
                f"{framework_name}, campaign {campaign_id}: "
                "non-finite evaluation scores."
            )

        threshold = float(
            np.quantile(
                training_scores,
                selection_quantile,
            )
        )

        fade_decisions = (
            current_scores >= threshold
        )

        decision_frame = build_decision_frame(
            current=current_data,
            framework_name=framework_name,
            scores=current_scores,
            thresholds=np.full(
                len(current_data),
                threshold,
                dtype=float,
            ),
            fade_decisions=fade_decisions,
        )

        decision_frames.append(
            decision_frame
        )

        print(
            f"{framework_name}, campaign {campaign_id}: "
            f"model={model_name}, "
            f"features={len(feature_columns)}, "
            f"trained={len(training_data):,}, "
            f"scored={len(current_data):,}, "
            f"threshold={threshold:.6f}, "
            f"coverage={fade_decisions.mean():.2%}"
        )

    decisions = pd.concat(
        decision_frames,
        ignore_index=True,
    )

    return decisions


def evaluate_framework_1_experiment(
    decisions,
):
    """Evaluate one or more Framework 1 experiment variants."""

    (
        summary,
        campaign_results,
        bootstrap_results,
        outcomes_available,
    ) = evaluate_frameworks(
        decisions
    )

    return {
        "decisions": decisions,
        "summary": summary,
        "campaign_results": campaign_results,
        "bootstrap_results": bootstrap_results,
        "outcomes_available": outcomes_available,
    }


framework_1_experiments = {}

print(
    "Framework 1 experiment utilities are ready."
)

print(
    "Reduced features:",
    len(FEATURE_COLUMNS),
)

print(
    "Full features:",
    len(FRAMEWORK_1_FULL_FEATURES),
)

Framework 1 experiment utilities are ready.
Reduced features: 22
Full features: 40


## Framework 1.2 - Tune selection threshold

In [65]:
# ======================================================
# Framework 1.2: Selection-quantile experiment
# ======================================================

FRAMEWORK_1_2_QUANTILES = [
    0.80,
    0.85,
    0.90,
    0.95,
]

framework_1_2_decision_frames = []


# ======================================================
# Run every candidate quantile
# ======================================================

for selection_quantile in (
    FRAMEWORK_1_2_QUANTILES
):
    framework_name = (
        "framework_1_2_q"
        f"{int(selection_quantile * 100)}"
    )

    decisions = run_framework_1_experiment(
        feature_data=historical_features,
        evaluation_campaigns=VALIDATION_CAMPAIGNS,
        framework_name=framework_name,
        feature_columns=FEATURE_COLUMNS,
        model_name="lightgbm",
        model_parameters=MODEL_PARAMETERS,
        selection_quantile=selection_quantile,
    )

    framework_1_2_decision_frames.append(
        decisions
    )


# ======================================================
# Combine candidate decisions
# ======================================================

if not framework_1_2_decision_frames:
    raise RuntimeError(
        "Framework 1.2 did not generate any decisions."
    )

framework_1_2_decisions = pd.concat(
    framework_1_2_decision_frames,
    ignore_index=True,
)

expected_frameworks = {
    (
        "framework_1_2_q"
        f"{int(quantile * 100)}"
    )
    for quantile in FRAMEWORK_1_2_QUANTILES
}

actual_frameworks = set(
    framework_1_2_decisions[
        "framework"
    ].unique()
)

if actual_frameworks != expected_frameworks:
    raise AssertionError(
        "Framework 1.2 candidate mismatch. "
        f"Found {sorted(actual_frameworks)}; "
        f"expected {sorted(expected_frameworks)}."
    )


# ======================================================
# Evaluate all candidate quantiles
# ======================================================

framework_1_experiments[
    "framework_1_2"
] = evaluate_framework_1_experiment(
    framework_1_2_decisions
)

framework_1_2_summary = (
    framework_1_experiments[
        "framework_1_2"
    ][
        "summary"
    ]
    .copy()
)

framework_1_2_campaign_results = (
    framework_1_experiments[
        "framework_1_2"
    ][
        "campaign_results"
    ]
    .copy()
)

framework_1_2_bootstrap_results = (
    framework_1_experiments[
        "framework_1_2"
    ][
        "bootstrap_results"
    ]
    .copy()
)


# ======================================================
# Attach the quantile value to the results
# ======================================================

quantile_lookup = {
    (
        "framework_1_2_q"
        f"{int(quantile * 100)}"
    ): quantile
    for quantile in FRAMEWORK_1_2_QUANTILES
}

framework_1_2_summary[
    "selection_quantile"
] = framework_1_2_summary[
    "framework"
].map(
    quantile_lookup
)

framework_1_2_campaign_results[
    "selection_quantile"
] = framework_1_2_campaign_results[
    "framework"
].map(
    quantile_lookup
)


# ======================================================
# Validate expected evaluation columns
# ======================================================

required_summary_columns = {
    "framework",
    "selection_quantile",
    "trades_evaluated",
    "trades_faded",
    "coverage",
    "lot_weighted_rp_per_lot",
    "ci_lower",
    "ci_upper",
    "positive_campaigns",
    "total_campaigns",
    "positive_campaign_share",
    "p_value_one_sided",
    "holm_adjusted_p",
    "holm_significant",
    "edge_above_zero",
    "ci_clears_zero",
    "campaign_share_pass",
    "passes_required_criteria",
}

missing_summary_columns = (
    required_summary_columns
    - set(framework_1_2_summary.columns)
)

if missing_summary_columns:
    raise ValueError(
        "Framework 1.2 summary is missing columns: "
        f"{sorted(missing_summary_columns)}"
    )


# ======================================================
# Rank the candidates
# ======================================================

framework_1_2_summary = (
    framework_1_2_summary.sort_values(
        [
            "passes_required_criteria",
            "positive_campaign_share",
            "ci_lower",
            "lot_weighted_rp_per_lot",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

framework_1_2_summary.insert(
    0,
    "validation_rank",
    np.arange(
        1,
        len(framework_1_2_summary) + 1,
    ),
)


# ======================================================
# Display the results
# ======================================================

summary_display_columns = [
    "validation_rank",
    "framework",
    "selection_quantile",
    "trades_evaluated",
    "trades_faded",
    "coverage",
    "lot_weighted_rp_per_lot",
    "ci_lower",
    "ci_upper",
    "positive_campaigns",
    "total_campaigns",
    "positive_campaign_share",
    "p_value_one_sided",
    "holm_adjusted_p",
    "holm_significant",
    "edge_above_zero",
    "ci_clears_zero",
    "campaign_share_pass",
    "passes_required_criteria",
]

print(
    "\nFramework 1.2 validation summary:"
)

display(
    framework_1_2_summary[
        summary_display_columns
    ]
)

print(
    "\nFramework 1.2 campaign-level results:"
)

display(
    framework_1_2_campaign_results.sort_values(
        [
            "selection_quantile",
            "campaign_id",
        ]
    ).reset_index(
        drop=True
    )
)


# ======================================================
# Identify the strongest validation candidate
# ======================================================

framework_1_2_best_candidate = (
    framework_1_2_summary.iloc[
        0
    ]
)

print(
    "\nHighest-ranked Framework 1.2 candidate:"
)

print(
    "Framework:",
    framework_1_2_best_candidate[
        "framework"
    ],
)

print(
    "Selection quantile:",
    framework_1_2_best_candidate[
        "selection_quantile"
    ],
)

print(
    "Lot-weighted RP/lot:",
    framework_1_2_best_candidate[
        "lot_weighted_rp_per_lot"
    ],
)

print(
    "95% CI:",
    (
        framework_1_2_best_candidate[
            "ci_lower"
        ],
        framework_1_2_best_candidate[
            "ci_upper"
        ],
    ),
)

print(
    "Positive campaign share:",
    framework_1_2_best_candidate[
        "positive_campaign_share"
    ],
)

print(
    "Passes required criteria:",
    framework_1_2_best_candidate[
        "passes_required_criteria"
    ],
)


# ======================================================
# Save Framework 1.2 outputs
# ======================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

framework_1_2_summary.to_csv(
    OUTPUT_DIR
    / "framework_1_2_quantile_summary.csv",
    index=False,
)

framework_1_2_campaign_results.to_csv(
    OUTPUT_DIR
    / "framework_1_2_quantile_campaign_results.csv",
    index=False,
)

framework_1_2_bootstrap_results.to_csv(
    OUTPUT_DIR
    / "framework_1_2_quantile_bootstrap_results.csv",
    index=False,
)

framework_1_2_decisions.to_csv(
    OUTPUT_DIR
    / "framework_1_2_quantile_decisions.csv",
    index=False,
)

print(
    "\nSaved Framework 1.2 outputs under:",
    OUTPUT_DIR,
)

framework_1_2_q80, campaign 53: model=lightgbm, features=22, trained=25,165, scored=1,063, threshold=14.303496, coverage=20.13%
framework_1_2_q80, campaign 54: model=lightgbm, features=22, trained=26,228, scored=1,511, threshold=16.265739, coverage=19.59%
framework_1_2_q80, campaign 55: model=lightgbm, features=22, trained=27,739, scored=1,222, threshold=16.147292, coverage=16.86%
framework_1_2_q80, campaign 56: model=lightgbm, features=22, trained=28,961, scored=1,732, threshold=13.740925, coverage=23.85%
framework_1_2_q80, campaign 57: model=lightgbm, features=22, trained=30,693, scored=1,256, threshold=12.844151, coverage=25.56%
framework_1_2_q80, campaign 58: model=lightgbm, features=22, trained=31,949, scored=1,263, threshold=8.740800, coverage=25.42%
framework_1_2_q80, campaign 59: model=lightgbm, features=22, trained=33,212, scored=1,490, threshold=9.892894, coverage=23.69%
framework_1_2_q80, campaign 60: model=lightgbm, features=22, trained=34,702, scored=1,575, threshold=11.45

,validation_rank,framework,selection_quantile,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,ci_lower,ci_upper,positive_campaigns,total_campaigns,positive_campaign_share,p_value_one_sided,holm_adjusted_p,holm_significant,edge_above_zero,ci_clears_zero,campaign_share_pass,passes_required_criteria
0,1,framework_1_2_q80,0.80,21355,5207,0.243830,-4.228316,-20.844504,12.757404,5,14,0.357143,0.683658,1.0,False,False,False,False,False
1,2,framework_1_2_q90,0.90,21355,1986,0.092999,-24.636248,-52.501451,4.430463,4,14,0.285714,0.959520,1.0,False,False,False,False,False
2,3,framework_1_2_q95,0.95,21355,909,0.042566,-49.575477,-96.522127,-3.804626,4,14,0.285714,0.982009,1.0,False,False,False,False,False
3,4,framework_1_2_q85,0.85,21355,3318,0.155373,-18.312221,-39.006918,3.162359,3,14,0.214286,0.956022,1.0,False,False,False,False,False



Framework 1.2 campaign-level results:


,framework,campaign_id,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,decision_threshold,selection_quantile
0,framework_1_2_q80,53,1063,214,0.201317,-18.707251,14.303496,0.80
1,framework_1_2_q80,54,1511,296,0.195897,-21.366327,16.265739,0.80
2,framework_1_2_q80,55,1222,206,0.168576,-42.155285,16.147292,0.80
3,framework_1_2_q80,56,1732,413,0.238453,-46.610079,13.740925,0.80
4,framework_1_2_q80,57,1256,321,0.255573,-1.690180,12.844151,0.80
5,framework_1_2_q80,58,1263,321,0.254157,-11.777702,8.740800,0.80
6,framework_1_2_q80,59,1490,353,0.236913,40.563872,9.892894,0.80
7,framework_1_2_q80,60,1575,300,0.190476,9.094347,11.459810,0.80
8,framework_1_2_q80,61,1573,451,0.286713,-4.719618,9.344818,0.80
9,framework_1_2_q80,62,1602,369,0.230337,-24.315608,9.547728,0.80



Highest-ranked Framework 1.2 candidate:
Framework: framework_1_2_q80
Selection quantile: 0.8
Lot-weighted RP/lot: -4.228316285459756
95% CI: (np.float64(-20.84450392571788), np.float64(12.75740419755809))
Positive campaign share: 0.35714285714285715
Passes required criteria: False

Saved Framework 1.2 outputs under: /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/outputs/stage4


## Framework 1.3 - Tune LightGBM hyperparameters

In [66]:
# ======================================================
# Framework 1.3: LightGBM hyperparameter experiment
# ======================================================

FRAMEWORK_1_3_CONFIGURATIONS = {
    "default": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "num_leaves": 15,
        "random_state": SEED,
        "verbosity": -1,
    },

    "regularized_small": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "num_leaves": 7,
        "max_depth": 4,
        "min_child_samples": 40,
        "reg_alpha": 1.0,
        "reg_lambda": 5.0,
        "random_state": SEED,
        "verbosity": -1,
    },

    "regularized_medium": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "num_leaves": 15,
        "max_depth": 5,
        "min_child_samples": 30,
        "reg_alpha": 1.0,
        "reg_lambda": 5.0,
        "random_state": SEED,
        "verbosity": -1,
    },

    "strong_regularization": {
        "n_estimators": 400,
        "learning_rate": 0.02,
        "num_leaves": 7,
        "max_depth": 4,
        "min_child_samples": 60,
        "reg_alpha": 5.0,
        "reg_lambda": 10.0,
        "random_state": SEED,
        "verbosity": -1,
    },

    "subsampled": {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "num_leaves": 15,
        "max_depth": 5,
        "min_child_samples": 30,
        "subsample": 0.80,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 1.0,
        "reg_lambda": 5.0,
        "random_state": SEED,
        "verbosity": -1,
    },
}

FRAMEWORK_1_3_QUANTILE = 0.90

framework_1_3_decision_frames = []

for (
    configuration_name,
    configuration_parameters,
) in FRAMEWORK_1_3_CONFIGURATIONS.items():

    framework_name = (
        "framework_1_3_"
        f"{configuration_name}"
    )

    decisions = run_framework_1_experiment(
        feature_data=historical_features,
        evaluation_campaigns=VALIDATION_CAMPAIGNS,
        framework_name=framework_name,
        feature_columns=FEATURE_COLUMNS,
        model_name="lightgbm",
        model_parameters=configuration_parameters,
        selection_quantile=FRAMEWORK_1_3_QUANTILE,
    )

    framework_1_3_decision_frames.append(
        decisions
    )

framework_1_3_decisions = pd.concat(
    framework_1_3_decision_frames,
    ignore_index=True,
)

framework_1_experiments[
    "framework_1_3"
] = evaluate_framework_1_experiment(
    framework_1_3_decisions
)

framework_1_3_summary = (
    framework_1_experiments[
        "framework_1_3"
    ][
        "summary"
    ]
    .sort_values(
        [
            "passes_required_criteria",
            "positive_campaign_share",
            "ci_lower",
            "lot_weighted_rp_per_lot",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nFramework 1.3 validation summary:"
)

display(
    framework_1_3_summary
)

print(
    "\nFramework 1.3 campaign results:"
)

display(
    framework_1_experiments[
        "framework_1_3"
    ][
        "campaign_results"
    ]
)

framework_1_3_default, campaign 53: model=lightgbm, features=22, trained=25,165, scored=1,063, threshold=60.439365, coverage=9.97%
framework_1_3_default, campaign 54: model=lightgbm, features=22, trained=26,228, scored=1,511, threshold=57.532725, coverage=10.46%
framework_1_3_default, campaign 55: model=lightgbm, features=22, trained=27,739, scored=1,222, threshold=58.682902, coverage=7.36%
framework_1_3_default, campaign 56: model=lightgbm, features=22, trained=28,961, scored=1,732, threshold=51.320125, coverage=8.43%
framework_1_3_default, campaign 57: model=lightgbm, features=22, trained=30,693, scored=1,256, threshold=39.850729, coverage=8.52%
framework_1_3_default, campaign 58: model=lightgbm, features=22, trained=31,949, scored=1,263, threshold=38.202976, coverage=9.82%
framework_1_3_default, campaign 59: model=lightgbm, features=22, trained=33,212, scored=1,490, threshold=35.634304, coverage=8.46%
framework_1_3_default, campaign 60: model=lightgbm, features=22, trained=34,702, s

,framework,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,ci_lower,ci_upper,positive_campaigns,total_campaigns,positive_campaign_share,p_value_one_sided,holm_adjusted_p,holm_significant,edge_above_zero,ci_clears_zero,campaign_share_pass,passes_required_criteria
0,framework_1_3_strong_regularization,21355,2030,0.095060,-4.516253,-32.304143,22.922818,9,14,0.642857,0.632684,1.0,False,False,False,True,False
1,framework_1_3_regularized_small,21355,2024,0.094779,-1.687615,-31.470395,25.623129,8,14,0.571429,0.542229,1.0,False,False,False,False,False
2,framework_1_3_subsampled,21355,1839,0.086116,-8.846083,-41.259200,24.799449,7,14,0.500000,0.713143,1.0,False,False,False,False,False
3,framework_1_3_default,21355,1986,0.092999,-24.636248,-52.501451,4.430463,4,14,0.285714,0.959520,1.0,False,False,False,False,False
4,framework_1_3_regularized_medium,21355,1944,0.091033,-28.672645,-55.990067,-0.519154,3,14,0.214286,0.978011,1.0,False,False,False,False,False



Framework 1.3 campaign results:


,framework,campaign_id,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,decision_threshold
0,framework_1_3_default,53,1063,106,0.099718,4.684009,60.439365
1,framework_1_3_default,54,1511,158,0.104567,-17.134658,57.532725
2,framework_1_3_default,55,1222,90,0.073650,-60.632994,58.682902
3,framework_1_3_default,56,1732,146,0.084296,-117.854269,51.320125
4,framework_1_3_default,57,1256,107,0.085191,-20.638974,39.850729
...,...,...,...,...,...,...,...
65,framework_1_3_subsampled,62,1602,110,0.068664,-12.042614,43.808623
66,framework_1_3_subsampled,63,1715,146,0.085131,28.062731,39.226747
67,framework_1_3_subsampled,64,1696,124,0.073113,48.631880,42.416514
68,framework_1_3_subsampled,65,1746,179,0.102520,-69.578216,41.167286


## Framework 1.4 - Alternative models

In [67]:
# ======================================================
# Framework 1.4: Alternative-regressor experiment
# ======================================================

FRAMEWORK_1_4_MODELS = {
    "lightgbm": {
        "model_name": "lightgbm",
        "model_parameters": MODEL_PARAMETERS,
    },

    "ridge": {
        "model_name": "ridge",
        "model_parameters": {
            "alpha": 10.0,
        },
    },

    "elastic_net": {
        "model_name": "elastic_net",
        "model_parameters": {
            "alpha": 0.1,
            "l1_ratio": 0.5,
            "max_iter": 10_000,
            "random_state": SEED,
        },
    },

    "extra_trees": {
        "model_name": "extra_trees",
        "model_parameters": {
            "n_estimators": 300,
            "min_samples_leaf": 10,
            "max_features": 0.8,
            "n_jobs": -1,
            "random_state": SEED,
        },
    },

    "random_forest": {
        "model_name": "random_forest",
        "model_parameters": {
            "n_estimators": 300,
            "min_samples_leaf": 10,
            "max_features": 0.8,
            "n_jobs": -1,
            "random_state": SEED,
        },
    },

    "hist_gradient_boosting": {
        "model_name": "hist_gradient_boosting",
        "model_parameters": {
            "learning_rate": 0.05,
            "max_iter": 200,
            "max_leaf_nodes": 15,
            "min_samples_leaf": 20,
            "l2_regularization": 1.0,
            "random_state": SEED,
        },
    },
}

FRAMEWORK_1_4_QUANTILE = 0.90

framework_1_4_decision_frames = []

for (
    candidate_name,
    candidate_specification,
) in FRAMEWORK_1_4_MODELS.items():

    framework_name = (
        "framework_1_4_"
        f"{candidate_name}"
    )

    decisions = run_framework_1_experiment(
        feature_data=historical_features,
        evaluation_campaigns=VALIDATION_CAMPAIGNS,
        framework_name=framework_name,
        feature_columns=FEATURE_COLUMNS,
        model_name=(
            candidate_specification[
                "model_name"
            ]
        ),
        model_parameters=(
            candidate_specification[
                "model_parameters"
            ]
        ),
        selection_quantile=FRAMEWORK_1_4_QUANTILE,
    )

    framework_1_4_decision_frames.append(
        decisions
    )

framework_1_4_decisions = pd.concat(
    framework_1_4_decision_frames,
    ignore_index=True,
)

framework_1_experiments[
    "framework_1_4"
] = evaluate_framework_1_experiment(
    framework_1_4_decisions
)

framework_1_4_summary = (
    framework_1_experiments[
        "framework_1_4"
    ][
        "summary"
    ]
    .sort_values(
        [
            "passes_required_criteria",
            "positive_campaign_share",
            "ci_lower",
            "lot_weighted_rp_per_lot",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nFramework 1.4 validation summary:"
)

display(
    framework_1_4_summary
)

print(
    "\nFramework 1.4 campaign results:"
)

display(
    framework_1_experiments[
        "framework_1_4"
    ][
        "campaign_results"
    ]
)

framework_1_4_lightgbm, campaign 53: model=lightgbm, features=22, trained=25,165, scored=1,063, threshold=60.439365, coverage=9.97%
framework_1_4_lightgbm, campaign 54: model=lightgbm, features=22, trained=26,228, scored=1,511, threshold=57.532725, coverage=10.46%
framework_1_4_lightgbm, campaign 55: model=lightgbm, features=22, trained=27,739, scored=1,222, threshold=58.682902, coverage=7.36%
framework_1_4_lightgbm, campaign 56: model=lightgbm, features=22, trained=28,961, scored=1,732, threshold=51.320125, coverage=8.43%
framework_1_4_lightgbm, campaign 57: model=lightgbm, features=22, trained=30,693, scored=1,256, threshold=39.850729, coverage=8.52%
framework_1_4_lightgbm, campaign 58: model=lightgbm, features=22, trained=31,949, scored=1,263, threshold=38.202976, coverage=9.82%
framework_1_4_lightgbm, campaign 59: model=lightgbm, features=22, trained=33,212, scored=1,490, threshold=35.634304, coverage=8.46%
framework_1_4_lightgbm, campaign 60: model=lightgbm, features=22, trained=3

,framework,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,ci_lower,ci_upper,positive_campaigns,total_campaigns,positive_campaign_share,p_value_one_sided,holm_adjusted_p,holm_significant,edge_above_zero,ci_clears_zero,campaign_share_pass,passes_required_criteria
0,framework_1_4_hist_gradient_boosting,21355,13847,0.648420,-3.600203,-11.793202,4.258642,6,14,0.428571,0.806097,1.0,False,False,False,False,False
1,framework_1_4_extra_trees,21355,788,0.036900,-10.556507,-65.463851,47.318085,6,14,0.428571,0.644178,1.0,False,False,False,False,False
2,framework_1_4_random_forest,21355,620,0.029033,-17.088644,-84.893226,49.796986,6,14,0.428571,0.686157,1.0,False,False,False,False,False
3,framework_1_4_elastic_net,21355,2594,0.121470,-13.617596,-29.790022,2.898317,4,14,0.285714,0.948026,1.0,False,False,False,False,False
4,framework_1_4_ridge,21355,2581,0.120862,-14.336379,-30.614306,1.924952,4,14,0.285714,0.954023,1.0,False,False,False,False,False
5,framework_1_4_lightgbm,21355,1986,0.092999,-24.636248,-52.501451,4.430463,4,14,0.285714,0.959520,1.0,False,False,False,False,False



Framework 1.4 campaign results:


,framework,campaign_id,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,decision_threshold
0,framework_1_4_lightgbm,53,1063,106,0.099718,4.684009,60.439365
1,framework_1_4_lightgbm,54,1511,158,0.104567,-17.134658,57.532725
2,framework_1_4_lightgbm,55,1222,90,0.073650,-60.632994,58.682902
3,framework_1_4_lightgbm,56,1732,146,0.084296,-117.854269,51.320125
4,framework_1_4_lightgbm,57,1256,107,0.085191,-20.638974,39.850729
...,...,...,...,...,...,...,...
79,framework_1_4_hist_gradient_boosting,62,1602,1495,0.933208,-22.521808,-6.326069
80,framework_1_4_hist_gradient_boosting,63,1715,1640,0.956268,-7.149286,-4.437465
81,framework_1_4_hist_gradient_boosting,64,1696,142,0.083726,-14.306565,4.240148
82,framework_1_4_hist_gradient_boosting,65,1746,153,0.087629,0.143346,11.556772


## Framework 1.5 - Full 40-feature set

In [68]:
# ======================================================
# Framework 1.5: Reduced versus full feature set
# ======================================================

missing_full_features = (
    set(FRAMEWORK_1_FULL_FEATURES)
    - set(historical_features.columns)
)

if missing_full_features:
    raise ValueError(
        "The Stage 4 feature table does not contain all "
        "40 full-set features: "
        f"{sorted(missing_full_features)}"
    )

FRAMEWORK_1_5_FEATURE_SETS = {
    "reduced_22": FEATURE_COLUMNS,
    "full_40": FRAMEWORK_1_FULL_FEATURES,
}

FRAMEWORK_1_5_QUANTILE = 0.90

framework_1_5_decision_frames = []

for (
    feature_set_name,
    feature_set_columns,
) in FRAMEWORK_1_5_FEATURE_SETS.items():

    framework_name = (
        "framework_1_5_"
        f"{feature_set_name}"
    )

    decisions = run_framework_1_experiment(
        feature_data=historical_features,
        evaluation_campaigns=VALIDATION_CAMPAIGNS,
        framework_name=framework_name,
        feature_columns=feature_set_columns,
        model_name="lightgbm",
        model_parameters=MODEL_PARAMETERS,
        selection_quantile=FRAMEWORK_1_5_QUANTILE,
    )

    framework_1_5_decision_frames.append(
        decisions
    )

framework_1_5_decisions = pd.concat(
    framework_1_5_decision_frames,
    ignore_index=True,
)

framework_1_experiments[
    "framework_1_5"
] = evaluate_framework_1_experiment(
    framework_1_5_decisions
)

framework_1_5_summary = (
    framework_1_experiments[
        "framework_1_5"
    ][
        "summary"
    ]
    .sort_values(
        [
            "passes_required_criteria",
            "positive_campaign_share",
            "ci_lower",
            "lot_weighted_rp_per_lot",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

print(
    "\nFramework 1.5 validation summary:"
)

display(
    framework_1_5_summary
)

print(
    "\nFramework 1.5 campaign results:"
)

display(
    framework_1_experiments[
        "framework_1_5"
    ][
        "campaign_results"
    ]
)

ValueError: The Stage 4 feature table does not contain all 40 full-set features: ['elapsed_active_minutes', 'ideas_started_past_15_minutes', 'ideas_started_past_30_minutes', 'ideas_started_past_60_minutes', 'past_amount_count', 'past_amount_mean', 'past_amount_std', 'past_idea_start_count', 'past_ideas_started_per_active_hour', 'past_losing_idea_count', 'past_winning_idea_count', 'post_loss_amount_increased', 'realized_balance_before_trade', 'realized_distance_to_profit_target', 'realized_return_before_trade', 'trades_opened_past_15_minutes', 'trades_opened_past_60_minutes']

## Consolidated validation comparison

In [ ]:
# ======================================================
# Framework 1 experiments: Consolidated comparison
# ======================================================

if not framework_1_experiments:
    raise RuntimeError(
        "No Framework 1 experiments have been evaluated."
    )

framework_1_comparison_frames = []

for (
    experiment_name,
    experiment_result,
) in framework_1_experiments.items():

    experiment_summary = (
        experiment_result[
            "summary"
        ]
        .copy()
    )

    experiment_summary.insert(
        0,
        "experiment_group",
        experiment_name,
    )

    framework_1_comparison_frames.append(
        experiment_summary
    )

framework_1_validation_comparison = pd.concat(
    framework_1_comparison_frames,
    ignore_index=True,
)

framework_1_validation_comparison = (
    framework_1_validation_comparison.sort_values(
        [
            "passes_required_criteria",
            "positive_campaign_share",
            "ci_lower",
            "lot_weighted_rp_per_lot",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

comparison_columns = [
    "experiment_group",
    "framework",
    "trades_evaluated",
    "trades_faded",
    "coverage",
    "lot_weighted_rp_per_lot",
    "ci_lower",
    "ci_upper",
    "positive_campaigns",
    "total_campaigns",
    "positive_campaign_share",
    "p_value_one_sided",
    "holm_adjusted_p",
    "passes_required_criteria",
]

available_comparison_columns = [
    column
    for column in comparison_columns
    if column
    in framework_1_validation_comparison.columns
]

print(
    "\nConsolidated Framework 1 validation comparison:"
)

display(
    framework_1_validation_comparison[
        available_comparison_columns
    ]
)

framework_1_validation_comparison.to_csv(
    OUTPUT_DIR
    / "framework_1_validation_experiments.csv",
    index=False,
)

print(
    "\nSaved comparison to:",
    OUTPUT_DIR
    / "framework_1_validation_experiments.csv",
)

# Framework 2 — Stage 3 fixed reference


In [51]:
def fit_stage3_fixed_reference(feature_data):
    """
    Train once on campaigns 33-52 using the corrected person-keyed
    feature pipeline, then freeze both the weights and q90 threshold.
    """
    training_data = feature_data.loc[
        feature_data["campaign_id"].isin(TRAIN_CAMPAIGNS)
    ].copy()

    actual_campaigns = set(
        training_data["campaign_id"].astype(int).unique()
    )

    if actual_campaigns != set(TRAIN_CAMPAIGNS):
        raise ValueError(
            "Stage 3 reference training data must contain "
            "exactly campaigns 33-52."
        )

    model = fit_regressor(training_data)

    threshold = calibrate_training_threshold(
        model,
        training_data,
    )

    metadata = {
        "framework": "stage3_fixed_reference",
        "framework_origin": "stage3_exp10b",
        "model_training": "retrained",
        "identity_pipeline": "person_keyed",
        "feature_pipeline": "corrected_stage4_pipeline",
        "training_campaigns": list(TRAIN_CAMPAIGNS),
        "training_rows": int(len(training_data)),
        "feature_columns": FEATURE_COLUMNS,
        "n_features": len(FEATURE_COLUMNS),
        "target": "reverse_profit_per_lot",
        "model_parameters": MODEL_PARAMETERS,
        "threshold_quantile": QUANTILE,
        "decision_threshold": threshold,
        "weights_policy": "fixed_after_campaign_52",
        "threshold_policy": "fixed_after_campaign_52",
    }

    return model, threshold, metadata


def run_stage3_fixed_reference(
    feature_data,
    evaluation_campaigns,
    model,
    threshold,
):
    """Apply the frozen campaigns 33-52 reference to later campaigns."""
    decision_frames = []

    for campaign_id in evaluation_campaigns:
        current = feature_data.loc[
            feature_data["campaign_id"] == campaign_id
        ].copy()

        if current.empty:
            raise ValueError(
                f"No feature rows for campaign {campaign_id}."
            )

        scores = np.asarray(
            model.predict(current[FEATURE_COLUMNS]),
            dtype=float,
        )

        decisions = scores >= threshold

        decision_frames.append(
            build_decision_frame(
                current=current,
                framework_name="stage3_fixed_reference",
                scores=scores,
                thresholds=np.full(len(current), threshold),
                fade_decisions=decisions,
            )
        )

        print(
            f"stage3_fixed_reference, campaign {campaign_id}: "
            f"scored={len(current):,}, "
            f"threshold={threshold:.6f}, "
            f"coverage={decisions.mean():.2%}"
        )

    return pd.concat(
        decision_frames,
        ignore_index=True,
    )


stage3_reference_model = None
stage3_reference_threshold = None
stage3_reference_metadata = None

if "stage3_fixed_reference" in ENABLED_FRAMEWORKS:
    (
        stage3_reference_model,
        stage3_reference_threshold,
        stage3_reference_metadata,
    ) = fit_stage3_fixed_reference(
        historical_features
    )

    print(
        "Stage 3 fixed reference fitted on campaigns 33-52."
    )
    print(
        "Fixed threshold:",
        f"{stage3_reference_threshold:.6f}",
    )


Stage 3 fixed reference fitted on campaigns 33-52.
Fixed threshold: 60.439365


# Framework 3 — walk-forward with stand-down filter


## Behavioural-state construction


In [52]:
# ======================================================
# Stand-down filter: Historical position sizing
# ======================================================

def person_closed_lot_median(data, trades):
    """Find each person's median lot among strictly earlier closed trades."""

    completed = trades.loc[
        trades["close_date_time"].notna()
        & trades["amount"].notna(),
        ["person_id", "close_date_time", "amount"],
    ].copy()

    completed = completed.sort_values("close_date_time")

    output = pd.Series(np.nan, index=data.index, dtype=float)

    for person, entries in data.groupby("person_id", sort=False):
        events = completed.loc[
            completed["person_id"].eq(person)
        ]

        if events.empty:
            continue

        event_times = pd.DatetimeIndex(events["close_date_time"])
        lots = events["amount"].to_numpy(dtype=float)

        for index, opened in entries["open_date_time"].items():
            cutoff = event_times.searchsorted(opened, side="left")

            if cutoff:
                output.at[index] = float(np.median(lots[:cutoff]))

    return output


In [53]:
# ======================================================
# Stand-down filter: Behavioural states
# ======================================================

def attach_filter_states(data, idea_features, trades):
    """Construct behavioural states using information available before entry."""

    data = data.copy()

    # --------------------------------------------------
    # 1. Size-up: Person-level history across campaigns
    # --------------------------------------------------

    data["person_past_closed_median_lot"] = person_closed_lot_median(
        data,
        trades,
    )

    data["size_up_ratio"] = (
        data["amount"] / data["person_past_closed_median_lot"]
    ).replace([np.inf, -np.inf], np.nan)

    # --------------------------------------------------
    # 2. Hot streak: Reset for every campaign
    # --------------------------------------------------

    ideas = idea_features.loc[
        idea_features["outcome_observed"].eq(True)
        & idea_features["idea_end_time"].notna(),
        [
            "campaign_id",
            "person_id",
            "idea_end_time",
            "total_net_profit",
            "idea_id",
        ],
    ].copy()

    ideas = ideas.sort_values(
        ["campaign_id", "person_id", "idea_end_time", "idea_id"]
    )

    events = []

    for (campaign_id, person_id), group in ideas.groupby(
        ["campaign_id", "person_id"],
        sort=False,
    ):
        streak = 0

        for close_time, simultaneous in group.groupby(
            "idea_end_time",
            sort=True,
        ):
            if simultaneous["total_net_profit"].gt(0).all():
                streak += len(simultaneous)
            else:
                streak = 0

            events.append({
                "campaign_id": campaign_id,
                "person_id": person_id,
                "streak_event_time": close_time,
                "hot_streak_completed_wins": streak,
            })

    if events:
        event_df = pd.DataFrame(events).sort_values(
            "streak_event_time"
        )

        data = pd.merge_asof(
            data.sort_values("open_date_time"),
            event_df,
            left_on="open_date_time",
            right_on="streak_event_time",
            by=["campaign_id", "person_id"],
            direction="backward",
            allow_exact_matches=False,
        )
    else:
        data["hot_streak_completed_wins"] = np.nan

    data["hot_streak_completed_wins"] = (
        data["hot_streak_completed_wins"].fillna(0)
    )

    # --------------------------------------------------
    # 3. Near target and desperate pace
    # --------------------------------------------------

    if PROFIT_TARGET_AMOUNT is not None:
        data["profit_still_needed"] = (
            PROFIT_TARGET_AMOUNT
            - data["realized_pnl_before_trade"]
        )

        data["near_target_remaining_profit"] = (
            data["profit_still_needed"]
        )

        deadline = pd.to_datetime(
            data["campaign_id"].map(CAMPAIGN_DEADLINES_UTC),
            utc=True,
            errors="coerce",
        )

        if deadline.isna().any():
            raise ValueError(
                "Missing campaign deadlines for desperate-pace calculation."
            )

        hours_remaining = (
            deadline - data["open_date_time"]
        ).dt.total_seconds() / 3600

        data["desperate_pace_profit_per_hour"] = (
            data["profit_still_needed"]
            / hours_remaining.where(hours_remaining > 0)
        )

    else:
        data["near_target_remaining_profit"] = np.nan
        data["desperate_pace_profit_per_hour"] = np.nan

    return data.drop(
        columns=["streak_event_time"],
        errors="ignore",
    )


In [ ]:
# ======================================================
# Framework 3: Filter-candidate correlation analysis
# ======================================================

from scipy.stats import spearmanr


CANDIDATE_FEATURES = {
    "size_up": (
        "size_up_ratio"
    ),
    "hot_streak": (
        "hot_streak_completed_wins"
    ),
    "near_target": (
        "near_target_remaining_profit"
    ),
    "desperate_pace": (
        "desperate_pace_profit_per_hour"
    ),
}


# ======================================================
# 1. Validate required functions and data
# ======================================================

if "attach_filter_states" not in globals():
    raise RuntimeError(
        "attach_filter_states() has not been defined. "
        "Run the Framework 3 behavioural-state utility "
        "cells before this correlation-analysis cell."
    )

if "run_walk_forward_baseline" not in globals():
    raise RuntimeError(
        "run_walk_forward_baseline() has not been defined. "
        "Run the Framework 1 definition cell first."
    )

required_feature_columns = {
    "trade_row_id",
    "campaign_id",
    "person_id",
    "amount",
    "reverse_profit",
    "reverse_profit_per_lot",
    "open_date_time",
}

missing_feature_columns = (
    required_feature_columns
    - set(historical_features.columns)
)

if missing_feature_columns:
    raise ValueError(
        "Historical feature data is missing columns: "
        f"{sorted(missing_feature_columns)}"
    )


# ======================================================
# 2. Generate or reuse baseline validation decisions
# ======================================================

reuse_existing_baseline = False

if "validation_baseline_decisions" in globals():
    existing_baseline_campaigns = set(
        validation_baseline_decisions[
            "campaign_id"
        ]
        .dropna()
        .astype(int)
        .unique()
    )

    expected_baseline_campaigns = set(
        VALIDATION_CAMPAIGNS
    )

    required_baseline_columns = {
        "trade_row_id",
        "campaign_id",
        "fade_decision",
    }

    reuse_existing_baseline = (
        existing_baseline_campaigns
        == expected_baseline_campaigns
        and required_baseline_columns.issubset(
            validation_baseline_decisions.columns
        )
    )

if reuse_existing_baseline:
    print(
        "Reusing existing walk-forward baseline "
        "validation decisions."
    )

    validation_baseline_decisions = (
        validation_baseline_decisions.copy()
    )

else:
    print(
        "Generating walk-forward baseline decisions "
        "for campaigns "
        f"{min(VALIDATION_CAMPAIGNS)}–"
        f"{max(VALIDATION_CAMPAIGNS)}..."
    )

    validation_baseline_decisions = (
        run_walk_forward_baseline(
            feature_data=historical_features,
            evaluation_campaigns=(
                VALIDATION_CAMPAIGNS
            ),
        )
    )


# ======================================================
# 3. Validate baseline decisions
# ======================================================

if validation_baseline_decisions.empty:
    raise ValueError(
        "Walk-forward baseline produced no "
        "validation decisions."
    )

if validation_baseline_decisions[
    "trade_row_id"
].duplicated().any():
    raise AssertionError(
        "Baseline validation decisions contain "
        "duplicate trade_row_id values."
    )

actual_validation_campaigns = set(
    validation_baseline_decisions[
        "campaign_id"
    ]
    .astype(int)
    .unique()
)

expected_validation_campaigns = set(
    VALIDATION_CAMPAIGNS
)

if (
    actual_validation_campaigns
    != expected_validation_campaigns
):
    raise ValueError(
        "Baseline validation campaign mismatch. "
        f"Found {sorted(actual_validation_campaigns)}; "
        f"expected {sorted(expected_validation_campaigns)}."
    )


# ======================================================
# 4. Construct filter states for validation trades
# ======================================================

validation_filter_features = (
    historical_features.loc[
        historical_features[
            "campaign_id"
        ].isin(
            VALIDATION_CAMPAIGNS
        )
    ]
    .copy()
)

validation_filter_states = (
    attach_filter_states(
        data=validation_filter_features,
        idea_features=(
            stage1_historical[
                "idea_features"
            ]
        ),
        trades=(
            stage1_historical[
                "trades"
            ]
        ),
    )
)

if validation_filter_states[
    "trade_row_id"
].duplicated().any():
    raise AssertionError(
        "Filter-state table contains duplicate "
        "trade_row_id values."
    )

missing_candidate_features = (
    set(CANDIDATE_FEATURES.values())
    - set(validation_filter_states.columns)
)

if missing_candidate_features:
    raise ValueError(
        "Filter-state construction did not create "
        "the following candidate features: "
        f"{sorted(missing_candidate_features)}"
    )


# ======================================================
# 5. Attach the baseline fade gate
# ======================================================

baseline_gate = (
    validation_baseline_decisions[
        [
            "trade_row_id",
            "fade_decision",
        ]
    ]
    .rename(
        columns={
            "fade_decision":
                "baseline_fade_decision"
        }
    )
)

validation_filter_states = (
    validation_filter_states.merge(
        baseline_gate,
        on="trade_row_id",
        how="left",
        validate="one_to_one",
    )
)

if validation_filter_states[
    "baseline_fade_decision"
].isna().any():
    missing_gate_rows = int(
        validation_filter_states[
            "baseline_fade_decision"
        ]
        .isna()
        .sum()
    )

    raise AssertionError(
        f"{missing_gate_rows} validation trades are "
        "missing baseline fade decisions."
    )

validation_filter_states[
    "baseline_fade_decision"
] = (
    validation_filter_states[
        "baseline_fade_decision"
    ]
    .astype(bool)
)


# ======================================================
# 6. Keep only baseline-gated validation trades
# ======================================================

gated_validation = (
    validation_filter_states.loc[
        validation_filter_states[
            "baseline_fade_decision"
        ]
    ]
    .copy()
)

if gated_validation.empty:
    raise ValueError(
        "No validation trades passed the "
        "walk-forward baseline gate."
    )

print(
    "\nBaseline-gated validation sample:"
)

print(
    "Trades:",
    f"{len(gated_validation):,}",
)

print(
    "People:",
    gated_validation[
        "person_id"
    ].nunique(),
)

print(
    "Campaigns:",
    gated_validation[
        "campaign_id"
    ].nunique(),
)


# ======================================================
# 7. Calculate candidate-to-RP/lot correlations
# ======================================================

correlation_results = []

for (
    candidate,
    feature,
) in CANDIDATE_FEATURES.items():

    subset = (
        gated_validation[
            [
                "campaign_id",
                "person_id",
                feature,
                "reverse_profit_per_lot",
            ]
        ]
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna(
            subset=[
                feature,
                "reverse_profit_per_lot",
                "person_id",
            ]
        )
        .copy()
    )

    number_of_trades = len(
        subset
    )

    number_of_people = (
        subset[
            "person_id"
        ]
        .nunique()
    )

    number_of_campaigns = (
        subset[
            "campaign_id"
        ]
        .nunique()
    )

    unique_feature_values = (
        subset[
            feature
        ]
        .nunique()
    )

    unique_outcome_values = (
        subset[
            "reverse_profit_per_lot"
        ]
        .nunique()
    )

    if (
        number_of_trades < 3
        or unique_feature_values < 2
        or unique_outcome_values < 2
    ):
        correlation = np.nan
        p_value = np.nan

    else:
        (
            correlation,
            p_value,
        ) = spearmanr(
            subset[
                feature
            ],
            subset[
                "reverse_profit_per_lot"
            ],
            nan_policy="omit",
        )

        correlation = float(
            correlation
        )

        p_value = float(
            p_value
        )

    # The filter is looking for behavioural states
    # associated with lower reverse profit per lot.
    if pd.isna(correlation):
        direction = (
            "Undetermined"
        )

    elif correlation < 0:
        direction = (
            "Higher values → lower RP/lot (>=)"
        )

    elif correlation > 0:
        direction = (
            "Lower values → lower RP/lot (<=)"
        )

    else:
        direction = (
            "No monotonic association"
        )

    correlation_results.append(
        {
            "candidate": candidate,
            "feature": feature,
            "trades": number_of_trades,
            "persons": number_of_people,
            "campaigns": number_of_campaigns,
            "spearman_rho": correlation,
            "naive_p_value": p_value,
            "suggested_direction": direction,
        }
    )


filter_correlation_results = pd.DataFrame(
    correlation_results
)


# ======================================================
# 8. Display correlation results
# ======================================================

print(
    "\nFilter-candidate correlation with "
    "reverse profit per lot:"
)

display(
    filter_correlation_results.round(
        {
            "spearman_rho": 4,
            "naive_p_value": 4,
        }
    )
)

print(
    "\nNote: naive_p_value treats trades as independent. "
    "Use the person-cluster bootstrap and adjusted "
    "filter screening to decide whether a rule is retained."
)

Reusing existing walk-forward baseline validation decisions.

Baseline-gated validation sample:
Trades: 1,986
People: 827
Campaigns: 14

Filter-candidate correlation with reverse profit per lot:


,candidate,feature,trades,persons,campaigns,spearman_rho,naive_p_value,suggested_direction
0,size_up,size_up_ratio,1986,827,14,-0.0379,0.0916,Higher values → lower RP/lot (>=)
1,hot_streak,hot_streak_completed_wins,1986,827,14,-0.0641,0.0043,Higher values → lower RP/lot (>=)
2,near_target,near_target_remaining_profit,1986,827,14,0.1011,0.0000,Lower values → lower RP/lot (<=)
3,desperate_pace,desperate_pace_profit_per_hour,1986,827,14,0.0805,0.0003,Lower values → lower RP/lot (<=)



Note: naive_p_value treats trades as independent. Use the person-cluster bootstrap and adjusted filter screening to decide whether a rule is retained.


In [55]:
CANDIDATE_FEATURES = {
    "size_up": "size_up_ratio",
    "hot_streak": "hot_streak_completed_wins",
    "near_target": "near_target_remaining_profit",
    "desperate_pace": "desperate_pace_profit_per_hour",
}

CANDIDATE_DIRECTIONS = {
    "size_up": "high",
    "hot_streak": "high",
    "near_target": "low",
    "desperate_pace": "low",
}

CUTOFF_QUANTILES = {
    "size_up": [0.75, 0.85, 0.95],
    "hot_streak": [0.75, 0.85, 0.95],
    "near_target": [0.10, 0.20, 0.30],
    "desperate_pace": [0.75, 0.85, 0.95],
}


def build_filter_cutoff_grid(data):
    """Derive candidate cutoffs without inspecting outcomes."""
    cutoff_grid = {}

    for candidate, feature in CANDIDATE_FEATURES.items():
        values = (
            data[feature]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if candidate in {"near_target", "desperate_pace"}:
            values = values.loc[values >= 0]

        if values.empty:
            cutoff_grid[candidate] = []
            continue

        cutoffs = np.quantile(
            values,
            CUTOFF_QUANTILES[candidate],
        )

        if candidate == "hot_streak":
            cutoffs = np.ceil(cutoffs).astype(int)
            cutoffs = cutoffs[cutoffs > 0]

        cutoff_grid[candidate] = sorted(set(cutoffs))

    return cutoff_grid


def develop_stand_down_rules(
    baseline_decisions,
    feature_data,
    stage1_tables,
):
    """Develop filter rules using validation campaigns 53-66 only."""
    validation_features = feature_data.loc[
        feature_data["campaign_id"].isin(VALIDATION_CAMPAIGNS)
    ].copy()

    state_data = attach_filter_states(
        validation_features,
        stage1_tables["idea_features"],
        stage1_tables["trades"],
    )

    gate_columns = baseline_decisions[
        ["trade_row_id", "fade_decision"]
    ].rename(columns={"fade_decision": "baseline_gate"})

    state_data = state_data.merge(
        gate_columns,
        on="trade_row_id",
        how="left",
        validate="one_to_one",
    )

    gated = state_data.loc[
        state_data["baseline_gate"].fillna(False)
    ].copy()

    cutoff_grid = build_filter_cutoff_grid(gated)

    number_of_comparisons = sum(
        len(cutoffs) for cutoffs in cutoff_grid.values()
    )

    if number_of_comparisons == 0:
        return pd.DataFrame(), {}

    adjusted_confidence = (
        1
        - (1 - CONFIDENCE_LEVEL)
        / number_of_comparisons
    )

    rows = []

    for candidate, cutoffs in cutoff_grid.items():
        feature = CANDIDATE_FEATURES[candidate]
        direction = CANDIDATE_DIRECTIONS[candidate]

        for cutoff in cutoffs:
            if direction == "high":
                mask = gated[feature].ge(cutoff)
            else:
                mask = gated[feature].le(cutoff)

            subset = gated.loc[
                mask.fillna(False)
            ].copy()

            observed_edge = lot_weighted_edge(subset)
            people = subset["person_id"].nunique()

            samples = (
                person_cluster_bootstrap(
                    subset,
                    n_bootstrap=BOOTSTRAP_ITERATIONS,
                    seed=SEED,
                )
                if people >= 2
                else np.array([], dtype=float)
            )

            ci_lower, ci_upper = percentile_bootstrap_ci(
                samples,
                confidence=CONFIDENCE_LEVEL,
            )

            adjusted_lower, adjusted_upper = (
                percentile_bootstrap_ci(
                    samples,
                    confidence=adjusted_confidence,
                )
            )

            qualifies = bool(
                len(subset) > 0
                and people >= 2
                and np.isfinite(observed_edge)
                and observed_edge < 0
                and adjusted_upper < 0
            )

            rows.append(
                {
                    "candidate": candidate,
                    "feature": feature,
                    "direction": direction,
                    "cutoff": float(cutoff),
                    "trades": len(subset),
                    "persons": people,
                    "campaigns": subset["campaign_id"].nunique(),
                    "lot_weighted_rp_per_lot": observed_edge,
                    "ci_lower": ci_lower,
                    "ci_upper": ci_upper,
                    "adjusted_ci_lower": adjusted_lower,
                    "adjusted_ci_upper": adjusted_upper,
                    "qualifies": qualifies,
                }
            )

    screen = pd.DataFrame(rows)

    qualifying = (
        screen.loc[screen["qualifies"]]
        .sort_values(
            ["candidate", "trades", "cutoff"],
            ascending=[True, False, True],
        )
        .drop_duplicates("candidate")
    )

    screen["retained"] = False
    screen.loc[qualifying.index, "retained"] = True

    retained_rules = {
        row.candidate: {
            "feature": row.feature,
            "direction": row.direction,
            "cutoff": float(row.cutoff),
        }
        for row in qualifying.itertuples(index=False)
    }

    return screen, retained_rules


def run_walk_forward_filtered(
    feature_data,
    evaluation_campaigns,
    stage1_tables,
    retained_rules,
):
    """Apply retained stand-down rules in front of the baseline gate."""
    if not retained_rules:
        raise ValueError(
            "walk_forward_filtered was enabled, but no "
            "stand-down rule passed the validation criterion. "
            "Remove it from ENABLED_FRAMEWORKS."
        )

    baseline = run_walk_forward_baseline(
        feature_data=feature_data,
        evaluation_campaigns=evaluation_campaigns,
        framework_name="walk_forward_filtered",
    )

    evaluation_features = feature_data.loc[
        feature_data["campaign_id"].isin(evaluation_campaigns)
    ].copy()

    states = attach_filter_states(
        evaluation_features,
        stage1_tables["idea_features"],
        stage1_tables["trades"],
    )

    state_columns = [
        "trade_row_id",
        *sorted(
            {
                rule["feature"]
                for rule in retained_rules.values()
            }
        ),
    ]

    filtered = baseline.merge(
        states[state_columns],
        on="trade_row_id",
        how="left",
        validate="one_to_one",
    )

    stand_down = pd.Series(
        False,
        index=filtered.index,
    )

    for rule in retained_rules.values():
        values = filtered[rule["feature"]]

        if rule["direction"] == "high":
            rule_mask = values.ge(rule["cutoff"])
        else:
            rule_mask = values.le(rule["cutoff"])

        stand_down = (
            stand_down
            | rule_mask.fillna(False)
        )

    filtered["stand_down"] = stand_down
    filtered["fade_decision"] = (
        filtered["fade_decision"]
        & ~filtered["stand_down"]
    )

    return filtered[
        [
            "trade_row_id",
            "campaign_id",
            "account_id",
            "person_id",
            "amount",
            "reverse_profit",
            "open_date_time",
            "framework",
            "score",
            "threshold",
            "fade_decision",
            "stand_down",
        ]
    ].copy()


# Validation evaluation


In [45]:
# ======================================================
# Initialize validation framework outputs
# ======================================================

validation_outputs = {}

# These are used only by the filtered framework.
filter_screen = pd.DataFrame()
retained_filter_rules = {}

print(
    "Validation output container initialized."
)

Validation output container initialized.


## Framework 1

In [ ]:
# ======================================================
# Validate Framework 1: Walk-forward baseline
# ======================================================

FRAMEWORK_NAME = "walk_forward_baseline"

if FRAMEWORK_NAME not in ENABLED_FRAMEWORKS:
    validation_outputs.pop(
        FRAMEWORK_NAME,
        None,
    )

    print(
        f"Skipped: {FRAMEWORK_NAME} is not enabled."
    )

else:
    validation_outputs[
        FRAMEWORK_NAME
    ] = run_walk_forward_baseline(
        feature_data=historical_features,
        evaluation_campaigns=VALIDATION_CAMPAIGNS,
    )

    print(
        f"Completed validation: {FRAMEWORK_NAME}"
    )

walk_forward_baseline, campaign 53: trained=25,165, scored=1,063, threshold=60.439365, coverage=9.97%
walk_forward_baseline, campaign 54: trained=26,228, scored=1,511, threshold=57.532725, coverage=10.46%
walk_forward_baseline, campaign 55: trained=27,739, scored=1,222, threshold=58.682902, coverage=7.36%
walk_forward_baseline, campaign 56: trained=28,961, scored=1,732, threshold=51.320125, coverage=8.43%
walk_forward_baseline, campaign 57: trained=30,693, scored=1,256, threshold=39.850729, coverage=8.52%
walk_forward_baseline, campaign 58: trained=31,949, scored=1,263, threshold=38.202976, coverage=9.82%
walk_forward_baseline, campaign 59: trained=33,212, scored=1,490, threshold=35.634304, coverage=8.46%
walk_forward_baseline, campaign 60: trained=34,702, scored=1,575, threshold=40.757560, coverage=8.44%
walk_forward_baseline, campaign 61: trained=36,277, scored=1,573, threshold=38.911996, coverage=10.49%
walk_forward_baseline, campaign 62: trained=37,850, scored=1,602, threshold=35.7

## Framework 2

In [ ]:
# ======================================================
# Validate Framework 2: Stage 3 fixed reference
# ======================================================

FRAMEWORK_NAME = "stage3_fixed_reference"

if FRAMEWORK_NAME not in ENABLED_FRAMEWORKS:
    validation_outputs.pop(
        FRAMEWORK_NAME,
        None,
    )

    print(
        f"Skipped: {FRAMEWORK_NAME} is not enabled."
    )

else:
    if stage3_reference_model is None:
        raise RuntimeError(
            "stage3_reference_model is unavailable. "
            "Run the Stage 3 fixed-reference training cell first."
        )

    if stage3_reference_threshold is None:
        raise RuntimeError(
            "stage3_reference_threshold is unavailable. "
            "Run the Stage 3 fixed-reference training cell first."
        )

    validation_outputs[
        FRAMEWORK_NAME
    ] = run_stage3_fixed_reference(
        feature_data=historical_features,
        evaluation_campaigns=VALIDATION_CAMPAIGNS,
        model=stage3_reference_model,
        threshold=stage3_reference_threshold,
    )

    print(
        f"Completed validation: {FRAMEWORK_NAME}"
    )

stage3_fixed_reference, campaign 53: scored=1,063, threshold=60.439365, coverage=9.97%
stage3_fixed_reference, campaign 54: scored=1,511, threshold=60.439365, coverage=9.60%
stage3_fixed_reference, campaign 55: scored=1,222, threshold=60.439365, coverage=7.86%
stage3_fixed_reference, campaign 56: scored=1,732, threshold=60.439365, coverage=8.89%
stage3_fixed_reference, campaign 57: scored=1,256, threshold=60.439365, coverage=8.28%
stage3_fixed_reference, campaign 58: scored=1,263, threshold=60.439365, coverage=8.00%
stage3_fixed_reference, campaign 59: scored=1,490, threshold=60.439365, coverage=8.99%
stage3_fixed_reference, campaign 60: scored=1,575, threshold=60.439365, coverage=8.63%
stage3_fixed_reference, campaign 61: scored=1,573, threshold=60.439365, coverage=8.65%
stage3_fixed_reference, campaign 62: scored=1,602, threshold=60.439365, coverage=7.12%
stage3_fixed_reference, campaign 63: scored=1,715, threshold=60.439365, coverage=7.93%
stage3_fixed_reference, campaign 64: scored

## Framework 3

In [ ]:
# ======================================================
# Validate Framework 3: Walk-forward filtered
# ======================================================

FRAMEWORK_NAME = "walk_forward_filtered"

if FRAMEWORK_NAME not in ENABLED_FRAMEWORKS:
    validation_outputs.pop(
        FRAMEWORK_NAME,
        None,
    )

    print(
        f"Skipped: {FRAMEWORK_NAME} is not enabled."
    )

else:
    if (
        "retained_filter_rules" not in globals()
        or retained_filter_rules is None
    ):
        raise RuntimeError(
            "retained_filter_rules is unavailable. "
            "Run the Framework 3 filter-development cell first."
        )

    if not retained_filter_rules:
        raise RuntimeError(
            "No stand-down filter rules were retained. "
            "Framework 3 cannot be evaluated as a distinct framework."
        )

    validation_outputs[
        FRAMEWORK_NAME
    ] = run_walk_forward_filtered(
        feature_data=historical_features,
        evaluation_campaigns=VALIDATION_CAMPAIGNS,
        stage1_tables=stage1_historical,
        retained_rules=retained_filter_rules,
    )

    print(
        f"Completed validation: {FRAMEWORK_NAME}"
    )

    print(
        "Retained filter rules:",
        retained_filter_rules,
    )

RuntimeError: No stand-down filter rules were retained. Framework C cannot be evaluated as a distinct framework.

In [49]:
# ======================================================
# Combine and evaluate enabled validation frameworks
# ======================================================

if not validation_outputs:
    raise RuntimeError(
        "No validation framework results are available. "
        "Run at least one enabled framework cell first."
    )

enabled_framework_set = set(
    ENABLED_FRAMEWORKS
)

completed_framework_set = set(
    validation_outputs
)

missing_frameworks = (
    enabled_framework_set
    - completed_framework_set
)

if missing_frameworks:
    raise RuntimeError(
        "The following enabled frameworks have not produced "
        "validation results: "
        f"{sorted(missing_frameworks)}. "
        "Run their validation cells or remove them from "
        "ENABLED_FRAMEWORKS."
    )

# Ignore stale results belonging to frameworks that are no
# longer enabled. This means individual framework cells do
# not all need to be rerun after changing the configuration.
active_validation_outputs = {
    framework_name: validation_outputs[
        framework_name
    ]
    for framework_name in ENABLED_FRAMEWORKS
}

validation_decisions = pd.concat(
    [
        active_validation_outputs[
            framework_name
        ]
        for framework_name
        in ENABLED_FRAMEWORKS
    ],
    ignore_index=True,
)

actual_frameworks = set(
    validation_decisions[
        "framework"
    ].unique()
)

if actual_frameworks != enabled_framework_set:
    raise AssertionError(
        "The combined validation decision table does not "
        "match ENABLED_FRAMEWORKS. "
        f"Found {sorted(actual_frameworks)}; "
        f"expected {sorted(enabled_framework_set)}."
    )

# Confirm that every framework evaluated the same campaigns.
framework_campaign_sets = {
    framework_name: set(
        framework_data[
            "campaign_id"
        ].astype(int).unique()
    )
    for framework_name, framework_data
    in validation_decisions.groupby(
        "framework"
    )
}

expected_validation_campaigns = set(
    VALIDATION_CAMPAIGNS
)

for framework_name, campaign_set in (
    framework_campaign_sets.items()
):
    if campaign_set != expected_validation_campaigns:
        raise AssertionError(
            f"{framework_name} did not evaluate exactly "
            "the validation campaigns. "
            f"Found {sorted(campaign_set)}; "
            f"expected {sorted(expected_validation_campaigns)}."
        )

(
    validation_summary,
    validation_campaign_results,
    validation_bootstrap_results,
    validation_outcomes_available,
) = evaluate_frameworks(
    validation_decisions
)

print(
    "\nValidation framework summary:"
)

display(
    validation_summary
)

# print(
#     "\nValidation campaign results:"
# )

# display(
#     validation_campaign_results
# )

  walk_forward_baseline: faded=1,986, coverage=9.30%, edge=-24.6362, 95% CI=[-52.5015, 4.4305], positive campaigns=4/14
  stage3_fixed_reference: faded=1,819, coverage=8.52%, edge=-14.7723, 95% CI=[-44.3374, 14.0809], positive campaigns=6/14

Validation framework summary:


,framework,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,ci_lower,ci_upper,positive_campaigns,total_campaigns,positive_campaign_share,p_value_one_sided,holm_adjusted_p,holm_significant,edge_above_zero,ci_clears_zero,campaign_share_pass,passes_required_criteria
0,walk_forward_baseline,21355,1986,0.092999,-24.636248,-52.501451,4.430463,4,14,0.285714,0.95952,1.0,False,False,False,False,False
1,stage3_fixed_reference,21355,1819,0.085179,-14.772259,-44.337435,14.080948,6,14,0.428571,0.83908,1.0,False,False,False,False,False


# Framework specification record


In [33]:
framework_specification = {
    "enabled_frameworks": ENABLED_FRAMEWORKS,
    "campaign_roles": {
        "train": list(TRAIN_CAMPAIGNS),
        "validation": list(VALIDATION_CAMPAIGNS),
        "sealed_evaluation": list(SEALED_CAMPAIGNS),
    },
    "data_directories": {
        "train": str(TRAIN_DIR),
        "validation": str(VALIDATION_DIR),
        "sealed": str(SEALED_DIR),
    },
    "person_key": (
        "SHA256 hash of normalized email after joining "
        "within campaign on campaign_id and account_id"
    ),
    "missing_email_policy": MISSING_EMAIL_POLICY,
    "frozen_features": FEATURE_COLUMNS,
    "model_parameters": MODEL_PARAMETERS,
    "model_target": "reverse_profit_per_lot",
    "threshold_quantile": QUANTILE,
    "walk_forward_rule": (
        "For campaign c, train on all available campaigns < c."
    ),
    "stage3_fixed_reference": stage3_reference_metadata,
    "retained_filter_rules": retained_filter_rules,
    "primary_metric": "lot_weighted_reverse_profit_per_lot",
    "confidence_level": CONFIDENCE_LEVEL,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "bootstrap_cluster": "person_id",
    "multiple_testing_method": "Holm",
    "minimum_positive_campaign_share": (
        MIN_POSITIVE_CAMPAIGN_SHARE
    ),
    "pass_bar": {
        "lot_weighted_rp_per_lot_above_zero": True,
        "confidence_interval_lower_bound_above_zero": True,
        "minimum_positive_campaign_share": (
            MIN_POSITIVE_CAMPAIGN_SHARE
        ),
    },
    "interpretation_note": (
        "Campaigns 67-82 have already been viewed. The originally "
        "frozen result is confirmatory; revised results on the same "
        "campaigns are exploratory."
    ),
}

display(
    pd.DataFrame(
        {
            "framework": ENABLED_FRAMEWORKS,
            "enabled": True,
        }
    )
)


,framework,enabled
0,walk_forward_baseline,True
1,stage3_fixed_reference,True


# Load sealed campaigns


In [41]:
sealed_trades_raw = load_trade_directory(
    SEALED_DIR / "User Trades",
    is_unseen=True,
)

sealed_users_raw = load_user_directory(
    SEALED_DIR / "User Data"
)

validate_campaign_range(
    sealed_trades_raw,
    SEALED_CAMPAIGNS,
    "Sealed trades",
)

validate_campaign_range(
    sealed_users_raw,
    SEALED_CAMPAIGNS,
    "Sealed User Data",
)

if sealed_trades_raw["close_date_time"].isna().any():
    raise ValueError(
        "Sealed trades contain missing close timestamps."
    )

if sealed_trades_raw["net_profit"].isna().any():
    raise ValueError(
        "Sealed trades contain missing net_profit values."
    )

if sealed_trades_raw["reverse_profit"].isna().any():
    raise ValueError(
        "Sealed trades contain missing reverse_profit values."
    )

if (
    sealed_trades_raw["amount"].isna().any()
    or sealed_trades_raw["amount"].le(0).any()
):
    raise ValueError(
        "Sealed trades contain missing or non-positive amounts."
    )

(
    sealed_identity_table,
    sealed_identity_audit,
    sealed_missing_email_summary,
    sealed_missing_user_summary,
) = audit_user_identities(
    users=sealed_users_raw,
    trades=sealed_trades_raw,
    expected_campaigns=SEALED_CAMPAIGNS,
    dataset_label="Sealed",
)

sealed_trades = attach_person_ids(
    sealed_trades_raw,
    sealed_identity_table,
)

combined_trades = pd.concat(
    [historical_trades, sealed_trades],
    ignore_index=True,
    sort=False,
)

expected_combined_campaigns = set(
    TRAIN_CAMPAIGNS
    + VALIDATION_CAMPAIGNS
    + SEALED_CAMPAIGNS
)

actual_combined_campaigns = set(
    combined_trades["campaign_id"].astype(int).unique()
)

if actual_combined_campaigns != expected_combined_campaigns:
    raise ValueError(
        "Combined trade data does not contain exactly "
        "campaigns 33-82."
    )

CAMPAIGN_DEADLINES_UTC = build_campaign_deadlines(
    combined_trades
)

stage1_all = build_stage1_tables_from_trades(
    combined_trades
)

full_features = build_model_features(
    stage1_all
)

if len(full_features) != len(combined_trades):
    raise AssertionError(
        "Feature engineering changed the combined trade-row count."
    )

missing_sealed_features = (
    set(FEATURE_COLUMNS) - set(full_features.columns)
)

if missing_sealed_features:
    raise AssertionError(
        "Combined feature table is missing frozen features: "
        f"{sorted(missing_sealed_features)}"
    )

if full_features["person_id"].isna().any():
    raise AssertionError(
        "Combined feature table contains missing person IDs."
    )

print(
    f"Prepared {len(full_features):,} combined feature rows "
    "for campaigns 33-82."
)



Scanning unseen trade directory:
  /Users/charltonsiaw/Desktop/Contrarian-Market-Strategy-Behavioural-Systems-Design/data/sealed/User Trades
Found 16 unseen files.
  [1/16] Processing: Campaign 67 Data 07 July 2026 XAUUSD only (1D).csv
  [2/16] Processing: Campaign 68 Data 10 July 2026 XAUUSD only (1D).csv
  [3/16] Processing: Campaign 69 Data 14 July 2026 XAUUSD only (1D).csv
  [4/16] Processing: Campaign 70 Data 17 July 2026 XAUUSD only (1D).csv
  [5/16] Processing: Campaign 71 Data 21 Jul 2026 XAUUSD only (1D).csv
  [6/16] Processing: Campaign 72 Data 24 Jul 2026 XAUUSD only (1D).csv
  [7/16] Processing: Campaign 73 Data 28 Jul 2026 XAUUSD only (1D).csv
  [8/16] Processing: Campaign 74 Data 31 Jul 2026 XAUUSD only (1D).csv
  [9/16] Processing: Campaign 75 Data 04 Aug 2026 XAUUSD only (1D).csv
  [10/16] Processing: Campaign 76 Data 07 Aug 2026 XAUUSD only (1D).csv
  [11/16] Processing: Campaign 77 Data 11 Aug 2026 XAUUSD only (1D).csv
  [12/16] Processing: Campaign 78 Data 14 Aug 20

,dataset,user_data_rows,unique_trading_accounts,duplicate_user_data_rows,account_ids_reused_by_different_emails,emails_using_multiple_account_ids,same_email_multiple_accounts_in_campaign,missing_email_user_data_rows,missing_email_accounts_with_trades,missing_email_accounts_without_trades,missing_email_affected_trade_rows,missing_user_record_accounts,missing_user_record_affected_trade_rows,email_based_account_mappings,unique_person_ids
0,Sealed,7820,4746,0,500,1431,21,0,0,0,0,3,15,4743,2044



Missing-email accounts by campaign:


,campaign_id,missing_email_accounts,accounts_with_trades,accounts_without_trades



Missing User Data accounts by campaign:


,campaign_id,affected_trade_rows,affected_accounts
0,74,3,1
1,78,11,1
2,82,1,1



Identity status counts:


,identity_status,account_mappings
0,email_based,4743
1,missing_user_record_fallback,3



Identity mapping completed: 4,746 trading accounts.
Person IDs attached successfully: 31,498 trades.
Prepared 78,018 combined feature rows for campaigns 33-82.


# Sealed evaluation


In [42]:
sealed_outputs = {}

if "walk_forward_baseline" in ENABLED_FRAMEWORKS:
    sealed_outputs[
        "walk_forward_baseline"
    ] = run_walk_forward_baseline(
        feature_data=full_features,
        evaluation_campaigns=SEALED_CAMPAIGNS,
    )

if "stage3_fixed_reference" in ENABLED_FRAMEWORKS:
    sealed_outputs[
        "stage3_fixed_reference"
    ] = run_stage3_fixed_reference(
        feature_data=full_features,
        evaluation_campaigns=SEALED_CAMPAIGNS,
        model=stage3_reference_model,
        threshold=stage3_reference_threshold,
    )

if "walk_forward_filtered" in ENABLED_FRAMEWORKS:
    sealed_outputs[
        "walk_forward_filtered"
    ] = run_walk_forward_filtered(
        feature_data=full_features,
        evaluation_campaigns=SEALED_CAMPAIGNS,
        stage1_tables=stage1_all,
        retained_rules=retained_filter_rules,
    )

sealed_decisions = pd.concat(
    sealed_outputs.values(),
    ignore_index=True,
)

expected_frameworks = set(ENABLED_FRAMEWORKS)
actual_frameworks = set(
    sealed_decisions["framework"].unique()
)

if actual_frameworks != expected_frameworks:
    raise AssertionError(
        "Sealed decision frameworks do not match "
        "ENABLED_FRAMEWORKS."
    )

framework_trade_sets = {
    framework_name: set(part["trade_row_id"])
    for framework_name, part
    in sealed_decisions.groupby("framework")
}

reference_trade_set = next(
    iter(framework_trade_sets.values())
)

for framework_name, trade_set in framework_trade_sets.items():
    if trade_set != reference_trade_set:
        raise AssertionError(
            f"{framework_name} evaluated a different trade population."
        )

print(
    f"Generated {len(sealed_decisions):,} "
    "framework/trade decision rows."
)


walk_forward_baseline, campaign 67: trained=46,520, scored=1,673, threshold=31.341518, coverage=8.07%
walk_forward_baseline, campaign 68: trained=48,193, scored=1,919, threshold=23.722838, coverage=10.11%
walk_forward_baseline, campaign 69: trained=50,112, scored=1,903, threshold=18.199538, coverage=8.51%
walk_forward_baseline, campaign 70: trained=52,015, scored=2,346, threshold=12.337556, coverage=7.97%
walk_forward_baseline, campaign 71: trained=54,361, scored=1,943, threshold=13.685024, coverage=9.98%
walk_forward_baseline, campaign 72: trained=56,304, scored=1,882, threshold=13.652582, coverage=8.93%
walk_forward_baseline, campaign 73: trained=58,186, scored=1,885, threshold=14.856912, coverage=8.91%
walk_forward_baseline, campaign 74: trained=60,071, scored=2,065, threshold=14.960331, coverage=9.93%
walk_forward_baseline, campaign 75: trained=62,136, scored=1,904, threshold=17.483967, coverage=9.24%
walk_forward_baseline, campaign 76: trained=64,040, scored=1,833, threshold=16.50

# Final metrics, Holm correction and pass criteria


In [43]:
(
    evaluation_summary,
    campaign_results,
    bootstrap_results,
    outcomes_available,
) = evaluate_frameworks(
    sealed_decisions
)

print("Final framework summary:")
display(evaluation_summary)

print("Campaign-level results:")
display(campaign_results)

primary_result = evaluation_summary.loc[
    evaluation_summary["framework"].eq(
        "walk_forward_baseline"
    )
]

if not primary_result.empty:
    primary_passed = bool(
        primary_result[
            "passes_required_criteria"
        ].iloc[0]
    )

    print(
        "Primary walk-forward baseline result:",
        "PASS" if primary_passed else "FAIL",
    )


  walk_forward_baseline: faded=3,469, coverage=11.01%, edge=-2.4494, 95% CI=[-21.0491, 17.2068], positive campaigns=6/16
  stage3_fixed_reference: faded=2,910, coverage=9.24%, edge=-4.3254, 95% CI=[-24.8959, 17.0626], positive campaigns=6/16
Final framework summary:


,framework,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,ci_lower,ci_upper,positive_campaigns,total_campaigns,positive_campaign_share,p_value_one_sided,holm_adjusted_p,holm_significant,edge_above_zero,ci_clears_zero,campaign_share_pass,passes_required_criteria
0,walk_forward_baseline,31498,3469,0.110134,-2.449398,-21.049081,17.206792,6,16,0.375,0.596702,1.0,False,False,False,False,False
1,stage3_fixed_reference,31498,2910,0.092387,-4.325435,-24.895931,17.062639,6,16,0.375,0.670165,1.0,False,False,False,False,False


Campaign-level results:


,framework,campaign_id,trades_evaluated,trades_faded,coverage,lot_weighted_rp_per_lot,decision_threshold
0,walk_forward_baseline,67,1673,135,0.080693,-20.539078,31.341518
1,walk_forward_baseline,68,1919,194,0.101094,-1.646792,23.722838
2,walk_forward_baseline,69,1903,162,0.085129,-27.810412,18.199538
3,walk_forward_baseline,70,2346,187,0.079710,41.084499,12.337556
4,walk_forward_baseline,71,1943,194,0.099846,54.862050,13.685024
5,walk_forward_baseline,72,1882,168,0.089267,26.201923,13.652582
6,walk_forward_baseline,73,1885,168,0.089125,45.568271,14.856912
7,walk_forward_baseline,74,2065,205,0.099274,-11.591725,14.960331
8,walk_forward_baseline,75,1904,176,0.092437,-22.797101,17.483967
9,walk_forward_baseline,76,1833,158,0.086197,0.290593,16.503422


Primary walk-forward baseline result: FAIL


# Diagnostics


In [ ]:
coverage_diagnostics = (
    sealed_decisions.groupby(
        ["framework", "campaign_id"],
        as_index=False,
    )
    .agg(
        trades_evaluated=("trade_row_id", "size"),
        trades_faded=("fade_decision", "sum"),
        coverage=("fade_decision", "mean"),
        threshold=("threshold", "first"),
    )
)

selected_person_concentration = (
    sealed_decisions.loc[
        sealed_decisions["fade_decision"]
    ]
    .groupby(["framework", "person_id"], as_index=False)
    .agg(
        reverse_profit=("reverse_profit", "sum"),
        amount=("amount", "sum"),
        faded_trades=("trade_row_id", "size"),
    )
)

selected_campaign_concentration = (
    sealed_decisions.loc[
        sealed_decisions["fade_decision"]
    ]
    .groupby(["framework", "campaign_id"], as_index=False)
    .agg(
        reverse_profit=("reverse_profit", "sum"),
        amount=("amount", "sum"),
        faded_trades=("trade_row_id", "size"),
    )
)

print("Coverage and threshold diagnostics:")
display(coverage_diagnostics)

print("Largest person-level selected exposures:")
display(
    selected_person_concentration.sort_values(
        ["framework", "amount"],
        ascending=[True, False],
    ).groupby("framework").head(10)
)


Coverage and threshold diagnostics:


,framework,campaign_id,trades_evaluated,trades_faded,coverage,threshold
0,stage3_fixed_reference,67,1673,140,0.083682,60.439365
1,stage3_fixed_reference,68,1919,164,0.085461,60.439365
2,stage3_fixed_reference,69,1903,155,0.081450,60.439365
3,stage3_fixed_reference,70,2346,172,0.073316,60.439365
4,stage3_fixed_reference,71,1943,168,0.086464,60.439365
5,stage3_fixed_reference,72,1882,138,0.073326,60.439365
6,stage3_fixed_reference,73,1885,154,0.081698,60.439365
7,stage3_fixed_reference,74,2065,153,0.074092,60.439365
8,stage3_fixed_reference,75,1904,164,0.086134,60.439365
9,stage3_fixed_reference,76,1833,126,0.068740,60.439365


Largest person-level selected exposures:


,framework,person_id,reverse_profit,amount,faded_trades
994,stage3_fixed_reference,email_f72e1c32a441e5848305cac3af63654ac8a86bd4...,-95.88,7.70,17
854,stage3_fixed_reference,email_d606cd25821e7a8336200a60ac1e662e7237fcf0...,-216.40,6.20,22
928,stage3_fixed_reference,email_e9a8afb9d7ba07b1a1e4cc5c11225f31542235d5...,-191.39,4.81,12
426,stage3_fixed_reference,email_6513cafb483140bd50de1a18906e70205e07b3cc...,-283.70,4.10,9
445,stage3_fixed_reference,email_69d813913efcab54b7d84d5b1d8a9b7a5d0f9366...,65.10,4.10,8
173,stage3_fixed_reference,email_27f7e25463e8d0024e648d2ce5514d5fbc8e4a3d...,153.44,3.87,12
987,stage3_fixed_reference,email_f637a0adde6b46f9fb7ac747725ff763ced14edc...,-121.13,3.59,8
363,stage3_fixed_reference,email_55d791c6c23aa6304feb302c6517a95db6425bd5...,-171.68,3.53,9
806,stage3_fixed_reference,email_c8a3a5d2ba6672220456b25feb5b46a0f32bbfaa...,83.40,3.20,9
594,stage3_fixed_reference,email_8fb06e3d88986f96ea55fb946c51f7b0777d36fd...,57.20,2.60,7


# Save models, decisions and results


In [ ]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

framework_specification_path = (
    OUTPUT_DIR / "framework_specification.json"
)

framework_specification_path.write_text(
    json.dumps(
        framework_specification,
        indent=2,
    ),
    encoding="utf-8",
)

identity_audit.to_csv(
    OUTPUT_DIR / "historical_identity_audit.csv",
    index=False,
)

sealed_identity_audit.to_csv(
    OUTPUT_DIR / "sealed_identity_audit.csv",
    index=False,
)

validation_decisions.to_csv(
    OUTPUT_DIR / "validation_decisions.csv",
    index=False,
)

validation_summary.to_csv(
    OUTPUT_DIR / "validation_summary.csv",
    index=False,
)

validation_campaign_results.to_csv(
    OUTPUT_DIR / "validation_campaign_results.csv",
    index=False,
)

sealed_decisions.to_csv(
    OUTPUT_DIR / "sealed_decisions.csv",
    index=False,
)

evaluation_summary.to_csv(
    OUTPUT_DIR / "evaluation_summary.csv",
    index=False,
)

campaign_results.to_csv(
    OUTPUT_DIR / "campaign_results.csv",
    index=False,
)

bootstrap_results.to_csv(
    OUTPUT_DIR / "bootstrap_results.csv",
    index=False,
)

coverage_diagnostics.to_csv(
    OUTPUT_DIR / "coverage_diagnostics.csv",
    index=False,
)

selected_person_concentration.to_csv(
    OUTPUT_DIR / "person_concentration.csv",
    index=False,
)

selected_campaign_concentration.to_csv(
    OUTPUT_DIR / "campaign_concentration.csv",
    index=False,
)

if not filter_screen.empty:
    filter_screen.to_csv(
        OUTPUT_DIR / "filter_screen.csv",
        index=False,
    )

if stage3_reference_model is not None:
    joblib.dump(
        stage3_reference_model,
        STAGE3_REFERENCE_MODEL_PATH,
    )

    STAGE3_REFERENCE_METADATA_PATH.write_text(
        json.dumps(
            stage3_reference_metadata,
            indent=2,
        ),
        encoding="utf-8",
    )

print("Outputs saved under:", OUTPUT_DIR)
print("Framework specification:", framework_specification_path)

if stage3_reference_model is not None:
    print("Stage 3 reference model:", STAGE3_REFERENCE_MODEL_PATH)
    print("Stage 3 reference metadata:", STAGE3_REFERENCE_METADATA_PATH)
